In [0]:
# Cell 0: Lakeflow runtime guard
# Purpose:
# - When Notebook 30 runs as the workflow_summary task, do NOT redeploy/reset/trigger the job.
# - Write a lightweight workflow runtime summary and exit cleanly.
# - This prevents old debug/deployment cells below from mutating the Lakeflow job structure.

from datetime import datetime, timezone
import uuid
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType,
)

# -------------------------------------------------------------------
# Safe widget helpers
# -------------------------------------------------------------------

def get_or_create_widget(name: str, default_value: str) -> str:
    try:
        return dbutils.widgets.get(name)
    except Exception:
        try:
            dbutils.widgets.text(name, default_value)
            return dbutils.widgets.get(name)
        except Exception:
            return default_value

execution_mode = get_or_create_widget("execution_mode", "workflow_summary_only").strip()
run_mode = get_or_create_widget("run_mode", "production").strip()
project_name = get_or_create_widget("project_name", "SupplySage AI").strip()
workflow_name = get_or_create_widget("workflow_name", "supplysage_lakeflow_daily_refresh").strip()
workflow_version = get_or_create_widget("workflow_version", "v1_lakeflow_jobs_orchestration").strip()
refresh_scope = get_or_create_widget("refresh_scope", "daily").strip()

GOLD_SCHEMA = "supplysage_gold"

print("Notebook 30 runtime guard")
print("=" * 100)
print("execution_mode:", execution_mode)
print("run_mode:", run_mode)
print("project_name:", project_name)
print("workflow_name:", workflow_name)
print("workflow_version:", workflow_version)
print("refresh_scope:", refresh_scope)

# -------------------------------------------------------------------
# Only allow old deployment cells if explicitly requested.
# Default behavior is workflow summary only.
# -------------------------------------------------------------------

if execution_mode != "deployment_admin":
    summary_started_at = datetime.now(timezone.utc)

    # These are the important tables produced by upstream Lakeflow tasks.
    # Do not include ui_* views here because Notebook 34 creates those after this task.
    tables_to_check = [
        f"{GOLD_SCHEMA}.gold_supplier_risk_scores",
        f"{GOLD_SCHEMA}.gold_sku_stockout_risk_scores",
        f"{GOLD_SCHEMA}.gold_rag_embeddings",
        f"{GOLD_SCHEMA}.gold_rag_retrieval_index",
        f"{GOLD_SCHEMA}.gold_chat_context_snapshots",
        f"{GOLD_SCHEMA}.gold_supplier_risk_agent_run_logs",
        f"{GOLD_SCHEMA}.gold_supplier_risk_agent_serving_health",
        f"{GOLD_SCHEMA}.gold_alert_events",
        f"{GOLD_SCHEMA}.gold_alert_inbox",
        f"{GOLD_SCHEMA}.gold_alert_delivery_log",
        f"{GOLD_SCHEMA}.gold_alerting_ui_breakdown",
    ]

    validation_rows = []

    for table_name in tables_to_check:
        try:
            df = spark.table(table_name)
            df.limit(1).collect()

            validation_rows.append({
                "table_name": table_name,
                "status": "PASS",
                "column_count": len(df.columns),
                "error": "",
            })
        except Exception as e:
            validation_rows.append({
                "table_name": table_name,
                "status": "WARN",
                "column_count": None,
                "error": str(e)[:500],
            })

    validation_schema = StructType([
        StructField("table_name", StringType(), False),
        StructField("status", StringType(), False),
        StructField("column_count", IntegerType(), True),
        StructField("error", StringType(), True),
    ])

    validation_df = spark.createDataFrame(validation_rows, schema=validation_schema)

    display(validation_df)

    pass_count = validation_df.filter("status = 'PASS'").count()
    warn_count = validation_df.filter("status != 'PASS'").count()

    summary_finished_at = datetime.now(timezone.utc)

    summary_table = f"{GOLD_SCHEMA}.gold_workflow_runtime_summary"

    summary_rows = [{
        "workflow_runtime_summary_id": f"workflow_summary_{summary_finished_at.strftime('%Y%m%d%H%M%S')}_{uuid.uuid4().hex[:8]}",
        "project_name": project_name,
        "workflow_name": workflow_name,
        "workflow_version": workflow_version,
        "run_mode": run_mode,
        "refresh_scope": refresh_scope,
        "execution_mode": execution_mode,
        "summary_status": "PASS" if warn_count == 0 else "WARN",
        "checked_table_count": len(tables_to_check),
        "pass_count": int(pass_count),
        "warn_count": int(warn_count),
        "summary_started_at": summary_started_at,
        "summary_finished_at": summary_finished_at,
        "notes": "Workflow summary-only guard executed. Deployment/debug cells were skipped intentionally.",
    }]

    summary_schema = StructType([
        StructField("workflow_runtime_summary_id", StringType(), False),
        StructField("project_name", StringType(), True),
        StructField("workflow_name", StringType(), True),
        StructField("workflow_version", StringType(), True),
        StructField("run_mode", StringType(), True),
        StructField("refresh_scope", StringType(), True),
        StructField("execution_mode", StringType(), True),
        StructField("summary_status", StringType(), True),
        StructField("checked_table_count", IntegerType(), True),
        StructField("pass_count", IntegerType(), True),
        StructField("warn_count", IntegerType(), True),
        StructField("summary_started_at", TimestampType(), True),
        StructField("summary_finished_at", TimestampType(), True),
        StructField("notes", StringType(), True),
    ])

    summary_df = spark.createDataFrame(summary_rows, schema=summary_schema)

    (
        summary_df
        .write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(summary_table)
    )

    print("\nWorkflow summary-only result")
    print("=" * 100)
    print("Summary table:", summary_table)
    print("PASS:", pass_count)
    print("WARN:", warn_count)
    print("Status:", "PASS" if warn_count == 0 else "WARN")
    print("✅ Notebook 30 completed safely in workflow_summary_only mode.")
    print("Skipping old deployment/debug cells below.")

    dbutils.notebook.exit(
        f"workflow_summary_only completed: PASS={pass_count}, WARN={warn_count}"
    )

print("deployment_admin mode enabled. Old deployment cells below will run.")
print("Use this mode only when intentionally creating/updating the Lakeflow job.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 1 (REPLACEMENT)
# Lakeflow Jobs config + workspace/API readiness
#
# WHAT CHANGED vs the broken version:
#   1. NOTEBOOK_PATHS now uses the ACTUAL workspace notebook names
#      (the copy-suffixed "(1)"/"(3)" names, "13_silver_external_risk_events",
#      "12_silver_synthetic_internal_tables", "Testing RAG", etc.).
#      Previously these used logical names that returned HTTP 404 in Cell 2.
#   2. "15_silver_domain_views" is removed from the silver task graph here,
#      at definition time, because it does not exist in the workspace.
#   3. Because the registry is correct up front, the separate "Cell 2B" patch
#      cell is no longer needed (you can delete it). Cell 2 now validates the
#      correct paths on the FIRST pass, so overall_validation_passed becomes
#      True and the Cell 3 assertion no longer halts the build.
#
#   >>> IMPORTANT: notebook names with spaces or "(1)" suffixes are fragile.
#       The clean long-term fix is to RENAME those workspace notebooks to plain
#       names (remove " (1)", " (3)", rename "Testing RAG" -> "26_rag_retrieval_test")
#       and then use the simple NOTEBOOK_PATHS block in the comment at the bottom.
# ─────────────────────────────────────────────────────────────

import json
import re
import time
import uuid
import platform
import requests
from datetime import datetime, timezone
from urllib.parse import urlparse

from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────
# Product / deployment config
# ─────────────────────────────────────────────────────────────

PROJECT_NAME = "SupplySage AI"
WORKFLOW_NAME = "supplysage_lakeflow_daily_refresh"
WORKFLOW_DISPLAY_NAME = "SupplySage AI - Lakeflow Daily Refresh"
WORKFLOW_VERSION = "v1_lakeflow_jobs_orchestration"

GOLD_SCHEMA = "supplysage_gold"
SILVER_SCHEMA = "supplysage_silver"
BRONZE_SCHEMA = "supplysage_bronze"

CREATED_BY = "Vigya"
NOTEBOOK_NAME = "30_lakeflow_jobs_orchestration_deployment"

# Safety flag: keep False until you explicitly decide to create/update the job.
CREATE_OR_UPDATE_JOB = True

JOB_TIMEZONE_ID = "America/Chicago"
JOB_QUARTZ_CRON = "0 0 6 * * ?"

JOB_BASE_PARAMETERS = {
    "project_name": PROJECT_NAME,
    "workflow_name": WORKFLOW_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "run_mode": "production",
    "refresh_scope": "daily",
    "external_ingestion_mode": "incremental",
    "agent_eval_mode": "smoke_test",
    "serving_health_mode": "health_only"
}

DEFAULT_MAX_RETRIES = 1
DEFAULT_MIN_RETRY_INTERVAL_MILLIS = 5 * 60 * 1000
DEFAULT_TIMEOUT_SECONDS = 60 * 60

# ─────────────────────────────────────────────────────────────
# Orchestration output tables
# ─────────────────────────────────────────────────────────────

WORKFLOW_JOB_MANIFEST_TABLE = f"{GOLD_SCHEMA}.gold_workflow_job_manifest"
WORKFLOW_RUN_STATUS_TABLE = f"{GOLD_SCHEMA}.gold_workflow_run_status"
WORKFLOW_TASK_HEALTH_TABLE = f"{GOLD_SCHEMA}.gold_workflow_task_health"
WORKFLOW_DEPLOYMENT_SUMMARY_TABLE = f"{GOLD_SCHEMA}.gold_workflow_deployment_summary"

REQUIRED_PRODUCT_TABLES = [
    f"{BRONZE_SCHEMA}.bronze_ext_ingestion_batch_manifest",
    f"{GOLD_SCHEMA}.gold_supplier_risk_scores",
    f"{GOLD_SCHEMA}.gold_sku_stockout_risk_scores",
    f"{GOLD_SCHEMA}.gold_rag_embeddings",
    f"{GOLD_SCHEMA}.gold_rag_retrieval_index",
    f"{GOLD_SCHEMA}.gold_chat_context_snapshots",
    f"{GOLD_SCHEMA}.gold_supplier_risk_agent_monitoring_metrics",
    f"{GOLD_SCHEMA}.gold_supplier_risk_agent_eval_results",
    f"{GOLD_SCHEMA}.gold_supplier_risk_agent_serving_health",
    f"{GOLD_SCHEMA}.gold_supplier_risk_agent_deployment_manifest"
]

# ─────────────────────────────────────────────────────────────
# Workspace context helpers
# ─────────────────────────────────────────────────────────────

def get_context_value(name: str):
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        if name == "notebook_path":
            return ctx.notebookPath().get()
        if name == "browser_host_name":
            return ctx.browserHostName().get()
        if name == "api_url":
            return ctx.apiUrl().get()
        if name == "workspace_id":
            return ctx.workspaceId().get()
        if name == "user":
            return ctx.userName().get()
    except Exception:
        return None
    return None


CURRENT_NOTEBOOK_PATH = get_context_value("notebook_path")
WORKSPACE_HOST = get_context_value("browser_host_name")
WORKSPACE_API_URL = get_context_value("api_url")
WORKSPACE_ID = get_context_value("workspace_id")
CURRENT_USER = get_context_value("user")

if CURRENT_NOTEBOOK_PATH:
    NOTEBOOK_BASE_PATH = "/".join(CURRENT_NOTEBOOK_PATH.split("/")[:-1])
else:
    NOTEBOOK_BASE_PATH = "/Users/vigya.awasthi@tamu.edu/supplysage-ai"

print("Workspace context:")
print("CURRENT_NOTEBOOK_PATH:", CURRENT_NOTEBOOK_PATH)
print("NOTEBOOK_BASE_PATH:", NOTEBOOK_BASE_PATH)
print("WORKSPACE_HOST:", WORKSPACE_HOST)
print("WORKSPACE_API_URL:", WORKSPACE_API_URL)
print("WORKSPACE_ID:", WORKSPACE_ID)
print("CURRENT_USER:", CURRENT_USER)

# ─────────────────────────────────────────────────────────────
# Notebook path registry — CORRECTED to actual workspace names
# ─────────────────────────────────────────────────────────────
# NOTE: The dict KEY is the stable logical key referenced by LAKEFLOW_TASK_GRAPH.
#       The VALUE is the real workspace path. Keys are intentionally left as the
#       original logical keys so the task graph below does not need to change.

NOTEBOOK_PATHS = {
    # Bronze external ingestion and validation
    "05_bronze_ingest_external_risk_sources_v0": f"{NOTEBOOK_BASE_PATH}/05_bronze_ingest_external_risk_sources_v0",
    "06_bronze_validate_external_risk_sources": f"{NOTEBOOK_BASE_PATH}/06_bronze_validate_external_risk_sources",

    # Silver layer
    "07_silver_m5_calendar": f"{NOTEBOOK_BASE_PATH}/07_silver_m5_calendar",
    "08_silver_m5_sell_prices": f"{NOTEBOOK_BASE_PATH}/08_silver_m5_sell_prices",
    "09_silver_m5_sales_daily": f"{NOTEBOOK_BASE_PATH}/09_silver_m5_sales_daily",
    "10_silver_retail_inventory": f"{NOTEBOOK_BASE_PATH}/10_silver_retail_inventory",
    "11_silver_dataco_orders_shipments": f"{NOTEBOOK_BASE_PATH}/11_silver_dataco_orders_shipments",
    # CORRECTED: real name is 12_silver_synthetic_internal_tables
    "12_silver_synthetic_supplier_tables": f"{NOTEBOOK_BASE_PATH}/12_silver_synthetic_internal_tables",
    # CORRECTED: real name is 13_silver_external_risk_events
    "13_silver_external_risk_events_evidence": f"{NOTEBOOK_BASE_PATH}/13_silver_external_risk_events",
    "14_silver_relationship_validation": f"{NOTEBOOK_BASE_PATH}/14_silver_relationship_validation",
    # 15_silver_domain_views intentionally NOT registered: it does not exist
    # in the workspace and is dropped from the task graph below.

    # Gold marts  (CORRECTED copy-suffix names from workspace discovery)
    "16_gold_dim_suppliers": f"{NOTEBOOK_BASE_PATH}/16_gold_dim_suppliers (1)",
    "17_gold_dim_products_skus": f"{NOTEBOOK_BASE_PATH}/17_gold_dim_products_skus (1)",
    "18_gold_supplier_sku_dependency_mart": f"{NOTEBOOK_BASE_PATH}/18_gold_supplier_sku_dependency_mart",
    "19_gold_inventory_stockout_feature_mart": f"{NOTEBOOK_BASE_PATH}/19_gold_inventory_stockout_feature_mart (1)",
    "20_gold_supplier_performance_mart": f"{NOTEBOOK_BASE_PATH}/20_gold_supplier_performance_mart (1)",
    "21_gold_external_risk_event_mart": f"{NOTEBOOK_BASE_PATH}/21_gold_external_risk_event_mart (3)",
    "22_gold_supplier_risk_scores": f"{NOTEBOOK_BASE_PATH}/22_gold_supplier_risk_scores",
    "23_gold_sku_stockout_risk_scores": f"{NOTEBOOK_BASE_PATH}/23_gold_sku_stockout_risk_scores (1)",
    "24_27_gold_rag_alerts_dashboards": f"{NOTEBOOK_BASE_PATH}/24_27_gold_rag_alerts_dashboards (1)",

    # RAG / context / agent
    "25_gold_rag_embeddings": f"{NOTEBOOK_BASE_PATH}/25_gold_rag_embeddings (3)",
    "25b_gold_chat_context_snapshots": f"{NOTEBOOK_BASE_PATH}/25b_gold_chat_context_snapshots",
    # CORRECTED: real name is "Testing RAG"
    "26_rag_retrieval_test": f"{NOTEBOOK_BASE_PATH}/Testing RAG",
    "27_langgraph_supplier_risk_agent": f"{NOTEBOOK_BASE_PATH}/27_langgraph_supplier_risk_agent",

    # Monitoring / serving / orchestration
    "28_mlflow_agent_monitoring_evaluation": f"{NOTEBOOK_BASE_PATH}/28_mlflow_agent_monitoring_evaluation",
    "29_agent_serving_deployment_wrapper": f"{NOTEBOOK_BASE_PATH}/29_agent_serving_deployment_wrapper",
    "30_lakeflow_jobs_orchestration_deployment": f"{NOTEBOOK_BASE_PATH}/30_lakeflow_jobs_orchestration_deployment",
}

print("\nRegistered notebook paths:")
for name, path in NOTEBOOK_PATHS.items():
    print(f"{name}: {path}")

# ─────────────────────────────────────────────────────────────
# Lakeflow Jobs task graph
# ─────────────────────────────────────────────────────────────
# 15_silver_domain_views is NOT included in silver_core_refresh.

LAKEFLOW_TASK_GRAPH = [
    {
        "task_key": "bronze_external_ingestion",
        "description": "Ingest external risk sources into Bronze Delta tables.",
        "notebook_key": "05_bronze_ingest_external_risk_sources_v0",
        "depends_on": [],
        "layer": "bronze",
        "critical": True
    },
    {
        "task_key": "bronze_external_validation",
        "description": "Validate latest clean Bronze ingestion batch.",
        "notebook_key": "06_bronze_validate_external_risk_sources",
        "depends_on": ["bronze_external_ingestion"],
        "layer": "bronze",
        "critical": True
    },
    {
        "task_key": "silver_core_refresh",
        "description": "Refresh Silver internal, external, and relationship tables.",
        "notebook_keys": [
            "07_silver_m5_calendar",
            "08_silver_m5_sell_prices",
            "09_silver_m5_sales_daily",
            "10_silver_retail_inventory",
            "11_silver_dataco_orders_shipments",
            "12_silver_synthetic_supplier_tables",
            "13_silver_external_risk_events_evidence",
            "14_silver_relationship_validation"
            # "15_silver_domain_views" removed: notebook does not exist in workspace
        ],
        "depends_on": ["bronze_external_validation"],
        "layer": "silver",
        "critical": True
    },
    {
        "task_key": "gold_marts_refresh",
        "description": "Refresh Gold dimensions, dependency marts, risk marts, and dashboard tables.",
        "notebook_keys": [
            "16_gold_dim_suppliers",
            "17_gold_dim_products_skus",
            "18_gold_supplier_sku_dependency_mart",
            "19_gold_inventory_stockout_feature_mart",
            "20_gold_supplier_performance_mart",
            "21_gold_external_risk_event_mart",
            "22_gold_supplier_risk_scores",
            "23_gold_sku_stockout_risk_scores",
            "24_27_gold_rag_alerts_dashboards"
        ],
        "depends_on": ["silver_core_refresh"],
        "layer": "gold",
        "critical": True
    },
    {
        "task_key": "rag_context_refresh",
        "description": "Refresh RAG embeddings, retrieval index, and supplier chat context snapshots.",
        "notebook_keys": [
            "25_gold_rag_embeddings",
            "25b_gold_chat_context_snapshots",
            "26_rag_retrieval_test"
        ],
        "depends_on": ["gold_marts_refresh"],
        "layer": "rag",
        "critical": True
    },
    {
        "task_key": "agent_runtime_smoke_test",
        "description": "Run supplier risk agent smoke test.",
        "notebook_key": "27_langgraph_supplier_risk_agent",
        "depends_on": ["rag_context_refresh"],
        "layer": "agent",
        "critical": True
    },
    {
        "task_key": "agent_monitoring_eval",
        "description": "Run MLflow monitoring and deterministic agent evaluation suite.",
        "notebook_key": "28_mlflow_agent_monitoring_evaluation",
        "depends_on": ["agent_runtime_smoke_test"],
        "layer": "monitoring",
        "critical": True
    },
    {
        "task_key": "serving_health_check",
        "description": "Run serving wrapper health checks and update deployment manifest.",
        "notebook_key": "29_agent_serving_deployment_wrapper",
        "depends_on": ["agent_monitoring_eval"],
        "layer": "serving",
        "critical": True
    },
    {
        "task_key": "workflow_summary",
        "description": "Record Lakeflow orchestration manifest and workflow deployment status.",
        "notebook_key": "30_lakeflow_jobs_orchestration_deployment",
        "depends_on": ["serving_health_check"],
        "layer": "orchestration",
        "critical": True
    }
]

print("\nLakeflow task graph:")
for task in LAKEFLOW_TASK_GRAPH:
    print(
        f"{task['task_key']} | layer={task['layer']} | "
        f"depends_on={task['depends_on']} | critical={task['critical']}"
    )

# ─────────────────────────────────────────────────────────────
# Jobs API readiness check
# ─────────────────────────────────────────────────────────────

def get_databricks_api_token():
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        return ctx.apiToken().get()
    except Exception:
        return None


def normalize_workspace_url(host_or_url):
    if host_or_url is None:
        return None
    value = str(host_or_url).strip()
    if value.startswith("https://"):
        return value.rstrip("/")
    return f"https://{value}".rstrip("/")


DATABRICKS_WORKSPACE_URL = normalize_workspace_url(WORKSPACE_HOST or WORKSPACE_API_URL)
DATABRICKS_API_TOKEN = get_databricks_api_token()

jobs_api_ready = bool(DATABRICKS_WORKSPACE_URL and DATABRICKS_API_TOKEN)

api_readiness = {
    "workspace_url_available": bool(DATABRICKS_WORKSPACE_URL),
    "api_token_available": bool(DATABRICKS_API_TOKEN),
    "jobs_api_ready": jobs_api_ready,
    "create_or_update_job_enabled": CREATE_OR_UPDATE_JOB
}

print("\nJobs API readiness:")
print(json.dumps(api_readiness, indent=2))

if not jobs_api_ready:
    print(
        "\nJobs API token is not available from this notebook context. "
        "We will still generate the Lakeflow Jobs config; deploy later via CLI/PAT/UI."
    )

# ─────────────────────────────────────────────────────────────
# Deployment metadata
# ─────────────────────────────────────────────────────────────

workflow_deployment_id = f"workflow_deploy_{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}_{uuid.uuid4().hex[:8]}"

workflow_metadata = {
    "workflow_deployment_id": workflow_deployment_id,
    "project_name": PROJECT_NAME,
    "workflow_name": WORKFLOW_NAME,
    "workflow_display_name": WORKFLOW_DISPLAY_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "job_timezone_id": JOB_TIMEZONE_ID,
    "job_quartz_cron": JOB_QUARTZ_CRON,
    "create_or_update_job": CREATE_OR_UPDATE_JOB,
    "jobs_api_ready": jobs_api_ready,
    "notebook_base_path": NOTEBOOK_BASE_PATH,
    "current_notebook_path": CURRENT_NOTEBOOK_PATH,
    "workspace_host": WORKSPACE_HOST,
    "workspace_id": WORKSPACE_ID,
    "current_user": CURRENT_USER,
    "python_version": platform.python_version(),
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}

print("\nWorkflow deployment metadata:")
print(json.dumps(workflow_metadata, indent=2))

print("\nCell 1 complete.")
print("Next: Cell 2 validates the (already-correct) notebook paths, required tables, and job graph.")

# ─────────────────────────────────────────────────────────────
# OPTIONAL CLEAN VERSION (use only AFTER renaming workspace notebooks)
# ─────────────────────────────────────────────────────────────
# If you rename the workspace notebooks to drop " (1)"/" (3)" suffixes and
# rename "Testing RAG" -> "26_rag_retrieval_test", replace the whole
# NOTEBOOK_PATHS dict with this generated one:
#
# _ALL_KEYS = [t.get("notebook_key") for t in LAKEFLOW_TASK_GRAPH if t.get("notebook_key")]
# for t in LAKEFLOW_TASK_GRAPH:
#     _ALL_KEYS.extend(t.get("notebook_keys", []))
# NOTEBOOK_PATHS = {k: f"{NOTEBOOK_BASE_PATH}/{k}" for k in sorted(set(_ALL_KEYS))}


In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Replacement Cell 2
# Validate patched notebook paths, product tables, and task graph integrity
# ─────────────────────────────────────────────────────────────

import json
import requests
from datetime import datetime, timezone

import pandas as pd
from pyspark.sql import functions as F

assert "NOTEBOOK_PATHS" in globals(), "NOTEBOOK_PATHS missing. Rerun Cell 1."
assert "LAKEFLOW_TASK_GRAPH" in globals(), "LAKEFLOW_TASK_GRAPH missing. Rerun Cell 1."
assert "REQUIRED_PRODUCT_TABLES" in globals(), "REQUIRED_PRODUCT_TABLES missing. Rerun Cell 1."
assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."

validation_started_at = datetime.now(timezone.utc)

# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────

def workspace_get_status(path: str) -> dict:
    if not DATABRICKS_WORKSPACE_URL or not DATABRICKS_API_TOKEN:
        return {
            "path": path,
            "exists": None,
            "object_type": None,
            "language": None,
            "error": "Workspace API not available."
        }

    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.0/workspace/get-status"
    headers = {"Authorization": f"Bearer {DATABRICKS_API_TOKEN}"}
    params = {"path": path}

    try:
        response = requests.get(url, headers=headers, params=params, timeout=30)

        if response.status_code == 200:
            payload = response.json()
            return {
                "path": path,
                "exists": True,
                "object_type": payload.get("object_type"),
                "language": payload.get("language"),
                "object_id": str(payload.get("object_id")) if payload.get("object_id") is not None else None,
                "error": None
            }

        return {
            "path": path,
            "exists": False,
            "object_type": None,
            "language": None,
            "object_id": None,
            "error": f"HTTP {response.status_code}: {response.text[:300]}"
        }

    except Exception as e:
        return {
            "path": path,
            "exists": False,
            "object_type": None,
            "language": None,
            "object_id": None,
            "error": str(e)
        }


def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False


def get_table_row_count(table_name: str):
    try:
        return spark.table(table_name).count()
    except Exception:
        return None


def get_table_columns(table_name: str):
    try:
        return spark.table(table_name).columns
    except Exception:
        return []


def get_best_timestamp_column(columns):
    timestamp_candidates = [
        "gold_created_at",
        "created_at",
        "updated_at",
        "snapshot_timestamp",
        "snapshot_date",
        "monitoring_logged_at",
        "finished_at",
        "health_finished_at",
        "run_finished_at",
        "saved_at"
    ]

    for col_name in timestamp_candidates:
        if col_name in columns:
            return col_name

    return None


def get_latest_timestamp(table_name: str, timestamp_column: str):
    try:
        row = (
            spark.table(table_name)
            .select(F.max(F.col(timestamp_column)).alias("latest_ts"))
            .collect()[0]
        )
        return row["latest_ts"]
    except Exception:
        return None


# ─────────────────────────────────────────────────────────────
# Collect notebook keys used by the actual task graph
# ─────────────────────────────────────────────────────────────

used_notebook_keys = []

for task in LAKEFLOW_TASK_GRAPH:
    if "notebook_key" in task:
        used_notebook_keys.append(task["notebook_key"])

    if "notebook_keys" in task:
        used_notebook_keys.extend(task["notebook_keys"])

used_notebook_keys = sorted(set(used_notebook_keys))

# ─────────────────────────────────────────────────────────────
# 1. Validate notebook paths used by task graph
# ─────────────────────────────────────────────────────────────

notebook_validation_rows = []

for notebook_key in used_notebook_keys:
    notebook_path = NOTEBOOK_PATHS.get(notebook_key)

    if notebook_path is None:
        notebook_validation_rows.append({
            "validation_type": "notebook_path",
            "asset_key": notebook_key,
            "asset_name": None,
            "exists": False,
            "status": "FAIL",
            "object_type": None,
            "language": None,
            "object_id": None,
            "row_count": None,
            "timestamp_column": None,
            "latest_timestamp": None,
            "error": "Notebook key missing from NOTEBOOK_PATHS."
        })
        continue

    status = workspace_get_status(notebook_path)

    notebook_validation_rows.append({
        "validation_type": "notebook_path",
        "asset_key": notebook_key,
        "asset_name": notebook_path,
        "exists": status.get("exists"),
        "status": "PASS" if status.get("exists") else "FAIL",
        "object_type": status.get("object_type"),
        "language": status.get("language"),
        "object_id": status.get("object_id"),
        "row_count": None,
        "timestamp_column": None,
        "latest_timestamp": None,
        "error": status.get("error")
    })

notebook_validation_pd = pd.DataFrame(notebook_validation_rows)

print("Notebook path validation:")
display(spark.createDataFrame(notebook_validation_pd))

missing_notebooks = notebook_validation_pd[notebook_validation_pd["status"] == "FAIL"]

if len(missing_notebooks) > 0:
    print("Missing notebook paths:")
    display(spark.createDataFrame(missing_notebooks))
else:
    print("All task-graph notebook paths exist.")

# ─────────────────────────────────────────────────────────────
# 2. Validate required product tables
# ─────────────────────────────────────────────────────────────

table_validation_rows = []

for table_name in REQUIRED_PRODUCT_TABLES:
    exists_flag = table_exists(table_name)
    row_count = get_table_row_count(table_name) if exists_flag else None
    columns = get_table_columns(table_name) if exists_flag else []
    timestamp_column = get_best_timestamp_column(columns)
    latest_timestamp = get_latest_timestamp(table_name, timestamp_column) if timestamp_column else None

    if not exists_flag:
        status = "FAIL"
        error = "Table does not exist or is not accessible."
    elif row_count is None or row_count == 0:
        status = "FAIL"
        error = "Table exists but has no rows."
    else:
        status = "PASS"
        error = None

    table_validation_rows.append({
        "validation_type": "required_table",
        "asset_key": table_name,
        "asset_name": table_name,
        "exists": bool(exists_flag),
        "status": status,
        "object_type": "DELTA_TABLE",
        "language": None,
        "object_id": None,
        "row_count": int(row_count) if row_count is not None else None,
        "timestamp_column": timestamp_column,
        "latest_timestamp": str(latest_timestamp) if latest_timestamp is not None else None,
        "error": error
    })

table_validation_pd = pd.DataFrame(table_validation_rows)

print("\nRequired product table validation:")
display(spark.createDataFrame(table_validation_pd))

failed_tables = table_validation_pd[table_validation_pd["status"] == "FAIL"]

if len(failed_tables) > 0:
    print("Failed required table checks:")
    display(spark.createDataFrame(failed_tables))
else:
    print("All required product tables exist and are non-empty.")

# ─────────────────────────────────────────────────────────────
# 3. Validate task graph integrity
# ─────────────────────────────────────────────────────────────

task_keys = [task["task_key"] for task in LAKEFLOW_TASK_GRAPH]
task_key_set = set(task_keys)

graph_validation_rows = []

duplicate_task_keys = sorted([key for key in task_key_set if task_keys.count(key) > 1])

graph_validation_rows.append({
    "validation_check": "unique_task_keys",
    "status": "PASS" if len(duplicate_task_keys) == 0 else "FAIL",
    "details": json.dumps({"duplicate_task_keys": duplicate_task_keys})
})

missing_dependency_refs = []

for task in LAKEFLOW_TASK_GRAPH:
    for dep in task.get("depends_on", []):
        if dep not in task_key_set:
            missing_dependency_refs.append({
                "task_key": task["task_key"],
                "missing_dependency": dep
            })

graph_validation_rows.append({
    "validation_check": "dependency_references_exist",
    "status": "PASS" if len(missing_dependency_refs) == 0 else "FAIL",
    "details": json.dumps({"missing_dependency_refs": missing_dependency_refs})
})

missing_notebook_refs = []

for task in LAKEFLOW_TASK_GRAPH:
    if "notebook_key" in task:
        if task["notebook_key"] not in NOTEBOOK_PATHS:
            missing_notebook_refs.append({
                "task_key": task["task_key"],
                "missing_notebook_key": task["notebook_key"]
            })

    if "notebook_keys" in task:
        for notebook_key in task["notebook_keys"]:
            if notebook_key not in NOTEBOOK_PATHS:
                missing_notebook_refs.append({
                    "task_key": task["task_key"],
                    "missing_notebook_key": notebook_key
                })

graph_validation_rows.append({
    "validation_check": "notebook_references_exist",
    "status": "PASS" if len(missing_notebook_refs) == 0 else "FAIL",
    "details": json.dumps({"missing_notebook_refs": missing_notebook_refs})
})

def has_cycle(tasks):
    graph = {task["task_key"]: task.get("depends_on", []) for task in tasks}
    visiting = set()
    visited = set()

    def visit(node):
        if node in visiting:
            return True

        if node in visited:
            return False

        visiting.add(node)

        for parent in graph.get(node, []):
            if visit(parent):
                return True

        visiting.remove(node)
        visited.add(node)
        return False

    for node in graph:
        if visit(node):
            return True

    return False

cycle_found = has_cycle(LAKEFLOW_TASK_GRAPH)

graph_validation_rows.append({
    "validation_check": "no_cycles",
    "status": "PASS" if not cycle_found else "FAIL",
    "details": json.dumps({"cycle_found": cycle_found})
})

expected_layer_order = [
    "bronze",
    "silver",
    "gold",
    "rag",
    "agent",
    "monitoring",
    "serving",
    "orchestration"
]

task_layers = [task.get("layer") for task in LAKEFLOW_TASK_GRAPH]
unknown_layers = sorted([layer for layer in set(task_layers) if layer not in expected_layer_order])

graph_validation_rows.append({
    "validation_check": "known_layers",
    "status": "PASS" if len(unknown_layers) == 0 else "FAIL",
    "details": json.dumps({
        "expected_layer_order": expected_layer_order,
        "unknown_layers": unknown_layers
    })
})

graph_validation_pd = pd.DataFrame(graph_validation_rows)

print("\nTask graph validation:")
display(spark.createDataFrame(graph_validation_pd))

failed_graph_checks = graph_validation_pd[graph_validation_pd["status"] == "FAIL"]

# ─────────────────────────────────────────────────────────────
# 4. Overall validation result
# ─────────────────────────────────────────────────────────────

validation_finished_at = datetime.now(timezone.utc)

notebook_pass_count = int((notebook_validation_pd["status"] == "PASS").sum())
notebook_fail_count = int((notebook_validation_pd["status"] == "FAIL").sum())

table_pass_count = int((table_validation_pd["status"] == "PASS").sum())
table_fail_count = int((table_validation_pd["status"] == "FAIL").sum())

graph_pass_count = int((graph_validation_pd["status"] == "PASS").sum())
graph_fail_count = int((graph_validation_pd["status"] == "FAIL").sum())

overall_validation_passed = (
    notebook_fail_count == 0
    and table_fail_count == 0
    and graph_fail_count == 0
)

validation_summary = {
    "workflow_deployment_id": workflow_deployment_id,
    "workflow_name": WORKFLOW_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "notebook_pass_count": notebook_pass_count,
    "notebook_fail_count": notebook_fail_count,
    "table_pass_count": table_pass_count,
    "table_fail_count": table_fail_count,
    "graph_pass_count": graph_pass_count,
    "graph_fail_count": graph_fail_count,
    "overall_validation_passed": overall_validation_passed,
    "validation_duration_seconds": (validation_finished_at - validation_started_at).total_seconds(),
    "validation_started_at": validation_started_at.isoformat(),
    "validation_finished_at": validation_finished_at.isoformat()
}

print("\nWorkflow validation summary:")
print(json.dumps(validation_summary, indent=2))

if not overall_validation_passed:
    print(
        "\nValidation did not fully pass. Review failed notebook/table/graph checks above before creating the Lakeflow Job."
    )
else:
    print("\nAll workflow deployment validations passed.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 2A
# Discover actual notebook paths in workspace folder
# ─────────────────────────────────────────────────────────────

import json
import re
import requests
import pandas as pd
from pyspark.sql import functions as F

assert "NOTEBOOK_BASE_PATH" in globals(), "NOTEBOOK_BASE_PATH missing. Rerun Cell 1."
assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."
assert "NOTEBOOK_PATHS" in globals(), "NOTEBOOK_PATHS missing. Rerun Cell 1."

headers = {"Authorization": f"Bearer {DATABRICKS_API_TOKEN}"}

def workspace_list(path: str) -> list:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.0/workspace/list"
    params = {"path": path}

    response = requests.get(url, headers=headers, params=params, timeout=30)

    if response.status_code != 200:
        raise ValueError(f"Workspace list failed for {path}: HTTP {response.status_code} {response.text[:500]}")

    return response.json().get("objects", [])


def normalize_name(value: str) -> str:
    """
    Normalize names for fuzzy matching:
    - lowercase
    - remove file-copy suffixes like (1), (3)
    - remove spaces, underscores, dashes, punctuation
    """
    if value is None:
        return ""

    value = str(value).lower()
    value = re.sub(r"\(\d+\)", "", value)
    value = re.sub(r"[^a-z0-9]", "", value)

    return value


# List objects directly under notebook base path
workspace_objects = workspace_list(NOTEBOOK_BASE_PATH)

workspace_rows = []

for obj in workspace_objects:
    path = obj.get("path")
    name = path.split("/")[-1] if path else None

    workspace_rows.append({
        "object_name": name,
        "path": path,
        "object_type": obj.get("object_type"),
        "language": obj.get("language"),
        "object_id": str(obj.get("object_id")) if obj.get("object_id") is not None else None,
        "normalized_name": normalize_name(name)
    })

workspace_objects_pd = pd.DataFrame(workspace_rows).sort_values("object_name")

print(f"Objects found under: {NOTEBOOK_BASE_PATH}")
display(spark.createDataFrame(workspace_objects_pd))

# Build matching candidates for registered notebook keys
actual_notebooks_pd = workspace_objects_pd[
    workspace_objects_pd["object_type"].isin(["NOTEBOOK"])
].copy()

registered_rows = []

for notebook_key, registered_path in NOTEBOOK_PATHS.items():
    registered_name = registered_path.split("/")[-1]
    registered_norm = normalize_name(registered_name)

    exact_match = actual_notebooks_pd[
        actual_notebooks_pd["path"] == registered_path
    ]

    normalized_match = actual_notebooks_pd[
        actual_notebooks_pd["normalized_name"] == registered_norm
    ]

    prefix_match = actual_notebooks_pd[
        actual_notebooks_pd["normalized_name"].str.startswith(registered_norm, na=False)
        | actual_notebooks_pd["normalized_name"].apply(lambda x: registered_norm.startswith(x) if isinstance(x, str) else False)
    ]

    if len(exact_match) > 0:
        match_status = "EXACT_PATH_MATCH"
        matched_path = exact_match.iloc[0]["path"]
        matched_name = exact_match.iloc[0]["object_name"]
    elif len(normalized_match) > 0:
        match_status = "NORMALIZED_NAME_MATCH"
        matched_path = normalized_match.iloc[0]["path"]
        matched_name = normalized_match.iloc[0]["object_name"]
    elif len(prefix_match) > 0:
        match_status = "POSSIBLE_PREFIX_MATCH"
        matched_path = prefix_match.iloc[0]["path"]
        matched_name = prefix_match.iloc[0]["object_name"]
    else:
        match_status = "NO_MATCH"
        matched_path = None
        matched_name = None

    registered_rows.append({
        "notebook_key": notebook_key,
        "registered_path": registered_path,
        "registered_name": registered_name,
        "match_status": match_status,
        "matched_name": matched_name,
        "matched_path": matched_path
    })

notebook_path_match_pd = pd.DataFrame(registered_rows)

print("\nNotebook registry match results:")
display(spark.createDataFrame(notebook_path_match_pd))

print("\nUnmatched registered notebooks:")
unmatched_pd = notebook_path_match_pd[notebook_path_match_pd["match_status"] == "NO_MATCH"]

if len(unmatched_pd) > 0:
    display(spark.createDataFrame(unmatched_pd))
else:
    print("All registered notebooks matched.")

# Patch NOTEBOOK_PATHS where we have exact, normalized, or prefix matches.
patched_count = 0

for row in notebook_path_match_pd.to_dict(orient="records"):
    if row["match_status"] in ["EXACT_PATH_MATCH", "NORMALIZED_NAME_MATCH", "POSSIBLE_PREFIX_MATCH"]:
        if row["matched_path"] and NOTEBOOK_PATHS[row["notebook_key"]] != row["matched_path"]:
            NOTEBOOK_PATHS[row["notebook_key"]] = row["matched_path"]
            patched_count += 1

print(f"\nPatched NOTEBOOK_PATHS entries: {patched_count}")

print("\nCurrent patched notebook paths:")
for key, path in NOTEBOOK_PATHS.items():
    print(f"{key}: {path}")

print("\nCell 2A complete.")
print("Next: rerun Replacement Cell 2 after reviewing unmatched notebooks.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 3 (COMPLETE)
# Validate, then build the SERVERLESS Lakeflow Jobs API payload.
#
# PART 1: validation guard — re-validates notebook paths if the Cell 2 flag is
#         stale/False, then asserts overall_validation_passed.
# PART 2: serverless payload build — defines the variables later cells need:
#         JOB_CLUSTER_KEY (label only), expanded_tasks, job_tasks,
#         lakeflow_job_payload, job_payload_summary.
#
# SERVERLESS: the payload must NOT define job_clusters and tasks must NOT carry
# job_cluster_key / new_cluster / existing_cluster_id, or Cell 7's
# serverless_payload_confirmed check will fail.
# ─────────────────────────────────────────────────────────────

import json
from datetime import datetime, timezone

import requests

assert "NOTEBOOK_PATHS" in globals(), "NOTEBOOK_PATHS missing. Rerun Cell 1."
assert "LAKEFLOW_TASK_GRAPH" in globals(), "LAKEFLOW_TASK_GRAPH missing. Rerun Cell 1."
assert "JOB_BASE_PARAMETERS" in globals(), "JOB_BASE_PARAMETERS missing. Rerun Cell 1."


# ─────────────────────────────────────────────────────────────
# PART 1 — Validation guard
# ─────────────────────────────────────────────────────────────

def _revalidate_notebook_paths():
    """Lightweight re-validation of notebook paths + graph integrity against
    the CURRENT NOTEBOOK_PATHS. Returns (passed: bool, problems: list)."""
    problems = []

    used_keys = []
    for task in LAKEFLOW_TASK_GRAPH:
        if "notebook_key" in task:
            used_keys.append(task["notebook_key"])
        if "notebook_keys" in task:
            used_keys.extend(task["notebook_keys"])
    used_keys = sorted(set(used_keys))

    for key in used_keys:
        if key not in NOTEBOOK_PATHS:
            problems.append(f"notebook_key '{key}' missing from NOTEBOOK_PATHS")

    ws_url = globals().get("DATABRICKS_WORKSPACE_URL")
    token = globals().get("DATABRICKS_API_TOKEN")
    if ws_url and token:
        headers = {"Authorization": f"Bearer {token}"}
        url = f"{ws_url}/api/2.0/workspace/get-status"
        for key in used_keys:
            path = NOTEBOOK_PATHS.get(key)
            if not path:
                continue
            try:
                resp = requests.get(url, headers=headers, params={"path": path}, timeout=30)
                if resp.status_code != 200:
                    problems.append(f"{key}: workspace path not found -> {path} (HTTP {resp.status_code})")
            except Exception as e:
                problems.append(f"{key}: workspace check error for {path}: {e}")
    else:
        print("NOTE: Jobs API token unavailable; skipped live workspace existence check.")

    return (len(problems) == 0, problems)


_flag = globals().get("overall_validation_passed")

if _flag is True:
    print("overall_validation_passed is already True (from Cell 2). Proceeding.")
else:
    print(
        "overall_validation_passed is not True "
        f"(value={_flag!r}). Re-validating against current NOTEBOOK_PATHS..."
    )
    _passed, _problems = _revalidate_notebook_paths()
    overall_validation_passed = _passed
    if _passed:
        print("Re-validation PASSED. overall_validation_passed set to True.")
    else:
        print("Re-validation FAILED. Problems:")
        for p in _problems:
            print(" -", p)

assert overall_validation_passed is True, (
    "Workflow validation failed. Fix NOTEBOOK_PATHS / task graph in Cell 1 "
    "(and rerun Cell 2) before building the job payload."
)


# ─────────────────────────────────────────────────────────────
# PART 2 — Serverless payload build
# ─────────────────────────────────────────────────────────────

# Cell 4's manifest references JOB_CLUSTER_KEY. Serverless tasks have no cluster,
# so we set a label-only value for the manifest record. It is NOT attached to any
# task in the payload (that would make Cell 7's serverless check fail).
JOB_CLUSTER_KEY = "serverless"


def _make_task_settings(task_key, notebook_key, depends_on_keys):
    """One Databricks Jobs 2.1/2.2 SERVERLESS task: no cluster keys at all."""
    return {
        "task_key": task_key,
        "depends_on": [{"task_key": d} for d in depends_on_keys],
        "notebook_task": {
            "notebook_path": NOTEBOOK_PATHS[notebook_key],
            "base_parameters": JOB_BASE_PARAMETERS,
            "source": "WORKSPACE"
        },
        # No job_cluster_key / new_cluster -> task runs on serverless compute.
        "max_retries": DEFAULT_MAX_RETRIES,
        "min_retry_interval_millis": DEFAULT_MIN_RETRY_INTERVAL_MILLIS,
        "timeout_seconds": DEFAULT_TIMEOUT_SECONDS,
    }


expanded_tasks = []
job_tasks = []

for graph_task in LAKEFLOW_TASK_GRAPH:
    logical_key = graph_task["task_key"]
    group_depends_on = list(graph_task.get("depends_on", []))

    if "notebook_key" in graph_task:
        nb_key = graph_task["notebook_key"]
        expanded_tasks.append({
            "task_key": logical_key,
            "logical_task_key": logical_key,
            "layer": graph_task["layer"],
            "critical": graph_task["critical"],
            "description": graph_task["description"],
            "notebook_key": nb_key,
            "depends_on": group_depends_on,
        })
        job_tasks.append(_make_task_settings(logical_key, nb_key, group_depends_on))

    elif "notebook_keys" in graph_task:
        prev_subtask_key = None
        for idx, nb_key in enumerate(graph_task["notebook_keys"], start=1):
            subtask_key = f"{logical_key}_{idx:02d}_{nb_key}"
            deps = group_depends_on if prev_subtask_key is None else [prev_subtask_key]
            expanded_tasks.append({
                "task_key": subtask_key,
                "logical_task_key": logical_key,
                "layer": graph_task["layer"],
                "critical": graph_task["critical"],
                "description": graph_task["description"],
                "notebook_key": nb_key,
                "depends_on": deps,
            })
            job_tasks.append(_make_task_settings(subtask_key, nb_key, deps))
            prev_subtask_key = subtask_key
    else:
        raise ValueError(f"Task '{logical_key}' has neither notebook_key nor notebook_keys.")

# Remap any dependency that points at a logical group key to that group's LAST sub-task.
_last_subtask_for_logical = {}
for graph_task in LAKEFLOW_TASK_GRAPH:
    logical_key = graph_task["task_key"]
    members = [t["task_key"] for t in expanded_tasks if t["logical_task_key"] == logical_key]
    if members:
        _last_subtask_for_logical[logical_key] = members[-1]

def _remap(dep_list):
    return [_last_subtask_for_logical.get(d, d) for d in dep_list]

for t in expanded_tasks:
    t["depends_on"] = _remap(t["depends_on"])
for jt in job_tasks:
    jt["depends_on"] = [{"task_key": d} for d in _remap([x["task_key"] for x in jt["depends_on"]])]

# ── Assemble the SERVERLESS Jobs API payload (no job_clusters) ──
lakeflow_job_payload = {
    "name": WORKFLOW_DISPLAY_NAME,
    "max_concurrent_runs": 1,
    "tags": {
        "project": "supplysage_ai",
        "workflow_version": WORKFLOW_VERSION,
        "managed_by": "notebook_30_lakeflow_orchestration"
    },
    "tasks": job_tasks,
    "schedule": {
        "quartz_cron_expression": JOB_QUARTZ_CRON,
        "timezone_id": JOB_TIMEZONE_ID,
        "pause_status": "PAUSED"
    }
}

job_payload_summary = {
    "job_name": lakeflow_job_payload["name"],
    "workflow_version": WORKFLOW_VERSION,
    "task_count": len(job_tasks),
    "logical_task_count": len(LAKEFLOW_TASK_GRAPH),
    "compute_mode": "serverless",
    "schedule": lakeflow_job_payload["schedule"],
    "task_keys": [t["task_key"] for t in job_tasks],
}

print(f"Built SERVERLESS Lakeflow payload: {len(job_tasks)} tasks "
      f"from {len(LAKEFLOW_TASK_GRAPH)} logical tasks. No job_clusters (serverless).")
for t in job_tasks:
    print(f"  - {t['task_key']}  depends_on={[d['task_key'] for d in t['depends_on']]}")

print("\n✅ Cell 3 complete: lakeflow_job_payload (serverless), expanded_tasks, job_tasks ready.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 4
# Save Lakeflow Jobs manifest + full API payload to Delta
# ─────────────────────────────────────────────────────────────

import json
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
    TimestampType
)

assert "lakeflow_job_payload" in globals(), "lakeflow_job_payload missing. Rerun Cell 3."
assert "expanded_tasks" in globals(), "expanded_tasks missing. Rerun Cell 3."
assert "job_tasks" in globals(), "job_tasks missing. Rerun Cell 3."
assert "workflow_deployment_id" in globals(), "workflow_deployment_id missing. Rerun Cell 1."

manifest_created_at = datetime.now(timezone.utc)

# ─────────────────────────────────────────────────────────────
# Job-level manifest row
# ─────────────────────────────────────────────────────────────

job_manifest_row = [{
    "workflow_deployment_id": workflow_deployment_id,
    "project_name": PROJECT_NAME,
    "workflow_name": WORKFLOW_NAME,
    "workflow_display_name": WORKFLOW_DISPLAY_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "job_name": lakeflow_job_payload.get("name"),
    "job_id": None,
    "job_created_or_updated": False,
    "create_or_update_job_enabled": bool(CREATE_OR_UPDATE_JOB),
    "jobs_api_ready": bool(jobs_api_ready),
    "task_count": int(len(job_tasks)),
    "schedule_quartz_cron": JOB_QUARTZ_CRON,
    "schedule_timezone_id": JOB_TIMEZONE_ID,
    "schedule_pause_status": lakeflow_job_payload.get("schedule", {}).get("pause_status"),
    "max_concurrent_runs": int(lakeflow_job_payload.get("max_concurrent_runs", 1)),
    "job_cluster_key": JOB_CLUSTER_KEY,
    "notebook_base_path": NOTEBOOK_BASE_PATH,
    "workspace_host": DATABRICKS_WORKSPACE_URL,
    "workspace_id": str(WORKSPACE_ID),
    "current_user": CURRENT_USER,
    "validation_passed": bool(overall_validation_passed),
    "deployment_status": "payload_generated_not_created",
    "lakeflow_job_payload_json": json.dumps(lakeflow_job_payload, indent=2, default=str),
    "job_payload_summary_json": json.dumps(job_payload_summary, indent=2, default=str),
    "created_at": manifest_created_at
}]

job_manifest_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("project_name", StringType(), True),
    StructField("workflow_name", StringType(), True),
    StructField("workflow_display_name", StringType(), True),
    StructField("workflow_version", StringType(), True),
    StructField("job_name", StringType(), True),
    StructField("job_id", StringType(), True),
    StructField("job_created_or_updated", BooleanType(), True),
    StructField("create_or_update_job_enabled", BooleanType(), True),
    StructField("jobs_api_ready", BooleanType(), True),
    StructField("task_count", IntegerType(), True),
    StructField("schedule_quartz_cron", StringType(), True),
    StructField("schedule_timezone_id", StringType(), True),
    StructField("schedule_pause_status", StringType(), True),
    StructField("max_concurrent_runs", IntegerType(), True),
    StructField("job_cluster_key", StringType(), True),
    StructField("notebook_base_path", StringType(), True),
    StructField("workspace_host", StringType(), True),
    StructField("workspace_id", StringType(), True),
    StructField("current_user", StringType(), True),
    StructField("validation_passed", BooleanType(), True),
    StructField("deployment_status", StringType(), True),
    StructField("lakeflow_job_payload_json", StringType(), True),
    StructField("job_payload_summary_json", StringType(), True),
    StructField("created_at", TimestampType(), True),
])

job_manifest_df = spark.createDataFrame(job_manifest_row, schema=job_manifest_schema)

(
    job_manifest_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(WORKFLOW_JOB_MANIFEST_TABLE)
)

print(f"Saved workflow job manifest to: {WORKFLOW_JOB_MANIFEST_TABLE}")

# ─────────────────────────────────────────────────────────────
# Task-level health / manifest rows
# ─────────────────────────────────────────────────────────────

task_manifest_rows = []

for idx, task in enumerate(expanded_tasks, start=1):
    task_manifest_rows.append({
        "workflow_deployment_id": workflow_deployment_id,
        "workflow_name": WORKFLOW_NAME,
        "workflow_version": WORKFLOW_VERSION,
        "task_order": int(idx),
        "task_key": task["task_key"],
        "logical_task_key": task["logical_task_key"],
        "layer": task["layer"],
        "critical": bool(task["critical"]),
        "description": task["description"],
        "notebook_key": task["notebook_key"],
        "notebook_path": NOTEBOOK_PATHS[task["notebook_key"]],
        "depends_on_json": json.dumps(task.get("depends_on", []), default=str),
        "job_cluster_key": JOB_CLUSTER_KEY,
        "max_retries": int(DEFAULT_MAX_RETRIES),
        "timeout_seconds": int(DEFAULT_TIMEOUT_SECONDS),
        "validation_status": "PASS",
        "created_at": manifest_created_at
    })

task_manifest_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("workflow_name", StringType(), True),
    StructField("workflow_version", StringType(), True),
    StructField("task_order", IntegerType(), True),
    StructField("task_key", StringType(), True),
    StructField("logical_task_key", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("critical", BooleanType(), True),
    StructField("description", StringType(), True),
    StructField("notebook_key", StringType(), True),
    StructField("notebook_path", StringType(), True),
    StructField("depends_on_json", StringType(), True),
    StructField("job_cluster_key", StringType(), True),
    StructField("max_retries", IntegerType(), True),
    StructField("timeout_seconds", IntegerType(), True),
    StructField("validation_status", StringType(), True),
    StructField("created_at", TimestampType(), True),
])

task_manifest_df = spark.createDataFrame(task_manifest_rows, schema=task_manifest_schema)

(
    task_manifest_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(WORKFLOW_TASK_HEALTH_TABLE)
)

print(f"Saved workflow task manifest to: {WORKFLOW_TASK_HEALTH_TABLE}")

# ─────────────────────────────────────────────────────────────
# Preview
# ─────────────────────────────────────────────────────────────

print("\nManifest saved for workflow_deployment_id:", workflow_deployment_id)
print("Deployment status: payload_generated_not_created")

display(
    spark.table(WORKFLOW_JOB_MANIFEST_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_name",
        "task_count",
        "schedule_quartz_cron",
        "schedule_timezone_id",
        "schedule_pause_status",
        "jobs_api_ready",
        "create_or_update_job_enabled",
        "job_created_or_updated",
        "deployment_status",
        "created_at"
    )
)

display(
    spark.table(WORKFLOW_TASK_HEALTH_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "task_order",
        "task_key",
        "logical_task_key",
        "layer",
        "notebook_key",
        "critical",
        "validation_status"
    )
    .orderBy("task_order")
)

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 5
# Dry-run Databricks Jobs deployer
# ─────────────────────────────────────────────────────────────

import json
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F

assert "lakeflow_job_payload" in globals(), "lakeflow_job_payload missing. Rerun Cell 3."
assert "jobs_api_ready" in globals(), "jobs_api_ready missing. Rerun Cell 1."
assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."

assert jobs_api_ready is True, "Jobs API is not ready. Cannot query Databricks Jobs API."

dry_run_started_at = datetime.now(timezone.utc)

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

def list_jobs_by_name(job_name: str) -> list:
    """
    List Databricks jobs and return jobs whose settings.name matches job_name.
    Uses Jobs API 2.1 list endpoint.
    """
    matching_jobs = []
    page_token = None

    while True:
        url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/list"

        params = {
            "limit": 25,
            "name": job_name
        }

        if page_token:
            params["page_token"] = page_token

        response = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=30
        )

        if response.status_code != 200:
            raise ValueError(
                f"Jobs list failed: HTTP {response.status_code} {response.text[:500]}"
            )

        payload = response.json()

        for job in payload.get("jobs", []):
            settings = job.get("settings", {}) or {}

            if settings.get("name") == job_name:
                matching_jobs.append(job)

        page_token = payload.get("next_page_token")

        if not page_token:
            break

    return matching_jobs


target_job_name = lakeflow_job_payload["name"]
existing_jobs = list_jobs_by_name(target_job_name)

if len(existing_jobs) == 0:
    proposed_action = "CREATE_JOB"
    existing_job_id = None
elif len(existing_jobs) == 1:
    proposed_action = "RESET_EXISTING_JOB"
    existing_job_id = str(existing_jobs[0].get("job_id"))
else:
    proposed_action = "AMBIGUOUS_MULTIPLE_JOBS_FOUND"
    existing_job_id = ",".join([str(job.get("job_id")) for job in existing_jobs])

dry_run_finished_at = datetime.now(timezone.utc)

dry_run_result = {
    "workflow_deployment_id": workflow_deployment_id,
    "job_name": target_job_name,
    "existing_job_count": len(existing_jobs),
    "existing_job_id": existing_job_id,
    "proposed_action": proposed_action,
    "create_or_update_job_enabled": bool(CREATE_OR_UPDATE_JOB),
    "jobs_api_ready": bool(jobs_api_ready),
    "would_create_or_update": bool(CREATE_OR_UPDATE_JOB and proposed_action in ["CREATE_JOB", "RESET_EXISTING_JOB"]),
    "dry_run_started_at": dry_run_started_at.isoformat(),
    "dry_run_finished_at": dry_run_finished_at.isoformat(),
    "dry_run_duration_seconds": (dry_run_finished_at - dry_run_started_at).total_seconds()
}

print("Lakeflow Jobs dry-run result:")
print(json.dumps(dry_run_result, indent=2))

if proposed_action == "AMBIGUOUS_MULTIPLE_JOBS_FOUND":
    print(
        "\nMultiple jobs with the same name were found. Do not auto-deploy until duplicates are resolved."
    )
elif CREATE_OR_UPDATE_JOB:
    print("\nCREATE_OR_UPDATE_JOB is True. The next deploy cell can create/update the Databricks job.")
else:
    print(
        "\nCREATE_OR_UPDATE_JOB is False. This was a safe dry run only. "
        "No Databricks job was created or updated."
    )

# Persist dry-run result by updating the latest manifest row status.
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, BooleanType, TimestampType
)

dry_run_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "job_name": target_job_name,
    "existing_job_count": int(len(existing_jobs)),
    "existing_job_id": str(existing_job_id) if existing_job_id is not None else None,
    "proposed_action": proposed_action,
    "create_or_update_job_enabled": bool(CREATE_OR_UPDATE_JOB),
    "would_create_or_update": bool(CREATE_OR_UPDATE_JOB and proposed_action in ["CREATE_JOB", "RESET_EXISTING_JOB"]),
    "dry_run_result_json": json.dumps(dry_run_result, indent=2, default=str),
    "dry_run_at": dry_run_finished_at
}]

dry_run_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("job_name", StringType(), True),
    StructField("existing_job_count", IntegerType(), True),
    StructField("existing_job_id", StringType(), True),
    StructField("proposed_action", StringType(), True),
    StructField("create_or_update_job_enabled", BooleanType(), True),
    StructField("would_create_or_update", BooleanType(), True),
    StructField("dry_run_result_json", StringType(), True),
    StructField("dry_run_at", TimestampType(), True),
])

dry_run_update_df = spark.createDataFrame(dry_run_rows, schema=dry_run_schema)

DRY_RUN_TABLE = f"{GOLD_SCHEMA}.gold_workflow_job_deploy_dry_run"

(
    dry_run_update_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(DRY_RUN_TABLE)
)

print(f"\nSaved dry-run deploy result to: {DRY_RUN_TABLE}")

display(
    spark.table(DRY_RUN_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_name",
        "existing_job_count",
        "existing_job_id",
        "proposed_action",
        "create_or_update_job_enabled",
        "would_create_or_update",
        "dry_run_at"
    )
    .orderBy(F.desc("dry_run_at"))
)

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 5A
# Save dry-run deploy result with explicit schema
# Fixes: [CANNOT_DETERMINE_TYPE] from null existing_job_id
# ─────────────────────────────────────────────────────────────

import json
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    BooleanType,
    TimestampType
)

assert "dry_run_result" in globals(), "dry_run_result missing. Rerun Cell 5."
assert "workflow_deployment_id" in globals(), "workflow_deployment_id missing. Rerun Cell 1."

DRY_RUN_TABLE = f"{GOLD_SCHEMA}.gold_workflow_job_deploy_dry_run"

dry_run_at = datetime.now(timezone.utc)

dry_run_rows = [{
    "workflow_deployment_id": str(dry_run_result.get("workflow_deployment_id")),
    "job_name": str(dry_run_result.get("job_name")),
    "existing_job_count": int(dry_run_result.get("existing_job_count") or 0),
    "existing_job_id": (
        str(dry_run_result.get("existing_job_id"))
        if dry_run_result.get("existing_job_id") is not None
        else None
    ),
    "proposed_action": str(dry_run_result.get("proposed_action")),
    "create_or_update_job_enabled": bool(dry_run_result.get("create_or_update_job_enabled")),
    "jobs_api_ready": bool(dry_run_result.get("jobs_api_ready")),
    "would_create_or_update": bool(dry_run_result.get("would_create_or_update")),
    "dry_run_result_json": json.dumps(dry_run_result, indent=2, default=str),
    "dry_run_at": dry_run_at
}]

dry_run_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("job_name", StringType(), True),
    StructField("existing_job_count", IntegerType(), True),
    StructField("existing_job_id", StringType(), True),
    StructField("proposed_action", StringType(), True),
    StructField("create_or_update_job_enabled", BooleanType(), True),
    StructField("jobs_api_ready", BooleanType(), True),
    StructField("would_create_or_update", BooleanType(), True),
    StructField("dry_run_result_json", StringType(), True),
    StructField("dry_run_at", TimestampType(), True),
])

dry_run_update_df = spark.createDataFrame(dry_run_rows, schema=dry_run_schema)

(
    dry_run_update_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(DRY_RUN_TABLE)
)

print(f"Saved dry-run deploy result to: {DRY_RUN_TABLE}")

display(
    spark.table(DRY_RUN_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_name",
        "existing_job_count",
        "existing_job_id",
        "proposed_action",
        "create_or_update_job_enabled",
        "jobs_api_ready",
        "would_create_or_update",
        "dry_run_at"
    )
    .orderBy(F.desc("dry_run_at"))
)

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 6A  (MISSING CELL — recreate it; place AFTER Cell 5/5A,
#                          BEFORE Cell 7 "verify")
# Build the SERVERLESS job payload and CREATE/RESET the Databricks Lakeflow Job.
#
# Produces the two variables Cell 7 (and Cells 8, 11) require:
#   serverless_job_payload  — the exact payload that was deployed
#   deployment_result       — dict with at least {"ok", "job_id", ...}
#
# This was the cell lost in editing. It mirrors the original serverless deploy
# that produced job 360345513847302 (compute_mode=serverless, 26 tasks, PAUSED).
# ─────────────────────────────────────────────────────────────

import json
import requests
from datetime import datetime, timezone

assert "lakeflow_job_payload" in globals(), "lakeflow_job_payload missing. Rerun Cell 3."
assert "dry_run_result" in globals(), "dry_run_result missing. Rerun Cell 5."
assert "jobs_api_ready" in globals(), "jobs_api_ready missing. Rerun Cell 1."
assert jobs_api_ready is True, "Jobs API is not ready."
assert "DATABRICKS_WORKSPACE_URL" in globals() and "DATABRICKS_API_TOKEN" in globals(), \
    "Workspace URL / token missing. Rerun Cell 1."

# The serverless payload IS the payload built in Cell 3 (no job_clusters,
# no job_cluster_key on tasks). We alias it under the name Cell 7 expects.
serverless_job_payload = lakeflow_job_payload

# Safety: confirm it really is serverless before deploying.
_classic = [
    t["task_key"] for t in serverless_job_payload.get("tasks", [])
    if ("job_cluster_key" in t or "new_cluster" in t or "existing_cluster_id" in t)
]
if _classic:
    raise ValueError(
        f"Payload is not serverless; these tasks carry cluster config: {_classic}. "
        "Use the serverless Cell 3 payload (no job_clusters)."
    )
if "job_clusters" in serverless_job_payload:
    raise ValueError("Serverless payload must not define job_clusters.")

# Real deploy switch for this cell.
DEPLOY_JOB_NOW = True

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

proposed_action = dry_run_result.get("proposed_action")
existing_job_id = dry_run_result.get("existing_job_id")

if proposed_action not in ["CREATE_JOB", "RESET_EXISTING_JOB"]:
    raise ValueError(
        f"Unsafe deploy action: {proposed_action}. Expected CREATE_JOB or RESET_EXISTING_JOB."
    )

if not CREATE_OR_UPDATE_JOB:
    raise ValueError("CREATE_OR_UPDATE_JOB is False. Set it True in Cell 1 to deploy.")

if not DEPLOY_JOB_NOW:
    raise ValueError("DEPLOY_JOB_NOW is False. Set True only when ready to deploy.")


def create_databricks_job(job_payload: dict) -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/create"
    response = requests.post(url, headers=headers, data=json.dumps(job_payload), timeout=120)
    if response.status_code not in [200, 201]:
        return {"ok": False, "status_code": response.status_code,
                "response_text": response.text[:2000], "job_id": None}
    payload = response.json()
    return {"ok": True, "status_code": response.status_code,
            "response_text": json.dumps(payload, indent=2),
            "job_id": str(payload.get("job_id")) if payload.get("job_id") is not None else None}


def reset_databricks_job(job_id: str, job_payload: dict) -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/reset"
    reset_payload = {"job_id": int(job_id), "new_settings": job_payload}
    response = requests.post(url, headers=headers, data=json.dumps(reset_payload), timeout=120)
    if response.status_code not in [200, 201]:
        return {"ok": False, "status_code": response.status_code,
                "response_text": response.text[:2000], "job_id": str(job_id)}
    return {"ok": True, "status_code": response.status_code,
            "response_text": response.text[:2000], "job_id": str(job_id)}


deploy_started_at = datetime.now(timezone.utc)

if proposed_action == "CREATE_JOB":
    deploy_api_result = create_databricks_job(serverless_job_payload)
    actual_action = "created"
else:  # RESET_EXISTING_JOB
    deploy_api_result = reset_databricks_job(existing_job_id, serverless_job_payload)
    actual_action = "reset"

deploy_finished_at = datetime.now(timezone.utc)

deployment_result = {
    "workflow_deployment_id": workflow_deployment_id,
    "job_name": serverless_job_payload["name"],
    "action": actual_action,
    "proposed_action": proposed_action,
    "ok": bool(deploy_api_result.get("ok")),
    "status_code": deploy_api_result.get("status_code"),
    "job_id": deploy_api_result.get("job_id") or (existing_job_id if actual_action == "reset" else None),
    "response_text": deploy_api_result.get("response_text"),
    "deploy_started_at": deploy_started_at.isoformat(),
    "deploy_finished_at": deploy_finished_at.isoformat(),
    "deploy_duration_seconds": (deploy_finished_at - deploy_started_at).total_seconds(),
}

print("Lakeflow serverless deploy result:")
print(json.dumps({k: v for k, v in deployment_result.items() if k != "response_text"}, indent=2))

if not deployment_result["ok"]:
    raise ValueError(
        f"Job {actual_action} failed: HTTP {deployment_result['status_code']}\n"
        f"{deploy_api_result.get('response_text')}"
    )

if not deployment_result["job_id"] or deployment_result["job_id"] == "None":
    raise ValueError("Deploy succeeded but no job_id was returned.")

print(f"\n✅ Cell 6A complete: job {actual_action}. JOB_ID = {deployment_result['job_id']}")
print("Next: Cell 7 verifies the deployed job; Cell 8 unpauses; Cell 11 adds the alerting task.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 6
# Create or update the actual Databricks Lakeflow Job
# ─────────────────────────────────────────────────────────────

import json
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    BooleanType,
    TimestampType
)

assert "lakeflow_job_payload" in globals(), "lakeflow_job_payload missing. Rerun Cell 3."
assert "dry_run_result" in globals(), "dry_run_result missing. Rerun Cell 5."
assert "jobs_api_ready" in globals(), "jobs_api_ready missing. Rerun Cell 1."
assert jobs_api_ready is True, "Jobs API is not ready."

# This is the real deployment switch for this cell.
DEPLOY_JOB_NOW = True

deploy_started_at = datetime.now(timezone.utc)

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

target_job_name = lakeflow_job_payload["name"]
proposed_action = dry_run_result.get("proposed_action")
existing_job_id = dry_run_result.get("existing_job_id")

if proposed_action not in ["CREATE_JOB", "RESET_EXISTING_JOB"]:
    raise ValueError(
        f"Unsafe deploy action: {proposed_action}. "
        "Expected CREATE_JOB or RESET_EXISTING_JOB."
    )

if not DEPLOY_JOB_NOW:
    raise ValueError("DEPLOY_JOB_NOW is False. Set to True only when ready to create/update the job.")

# ─────────────────────────────────────────────────────────────
# Deploy helpers
# ─────────────────────────────────────────────────────────────

def create_databricks_job(job_payload: dict) -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/create"

    response = requests.post(
        url,
        headers=headers,
        data=json.dumps(job_payload),
        timeout=120
    )

    if response.status_code not in [200, 201]:
        return {
            "ok": False,
            "status_code": response.status_code,
            "response_text": response.text[:2000],
            "job_id": None
        }

    payload = response.json()

    return {
        "ok": True,
        "status_code": response.status_code,
        "response_text": json.dumps(payload, indent=2),
        "job_id": str(payload.get("job_id")) if payload.get("job_id") is not None else None
    }


def reset_databricks_job(job_id: str, job_payload: dict) -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/reset"

    reset_payload = {
        "job_id": int(job_id),
        "new_settings": job_payload
    }

    response = requests.post(
        url,
        headers=headers,
        data=json.dumps(reset_payload),
        timeout=120
    )

    if response.status_code not in [200, 201]:
        return {
            "ok": False,
            "status_code": response.status_code,
            "response_text": response.text[:2000],
            "job_id": str(job_id)
        }

    return {
        "ok": True,
        "status_code": response.status_code,
        "response_text": response.text[:2000],
        "job_id": str(job_id)
    }


# ─────────────────────────────────────────────────────────────
# Execute deployment
# ─────────────────────────────────────────────────────────────

if proposed_action == "CREATE_JOB":
    deploy_api_result = create_databricks_job(lakeflow_job_payload)
    actual_action = "created"

elif proposed_action == "RESET_EXISTING_JOB":
    deploy_api_result = reset_databricks_job(existing_job_id, lakeflow_job_payload)
    actual_action = "reset"

else:
    raise ValueError(f"Unsupported proposed_action: {proposed_action}")

deploy_finished_at = datetime.now(timezone.utc)

job_id = deploy_api_result.get("job_id")
deployment_succeeded = bool(deploy_api_result.get("ok"))

if deployment_succeeded:
    deployment_status = "job_created_or_updated"
else:
    deployment_status = "job_deploy_failed"

deployment_result = {
    "workflow_deployment_id": workflow_deployment_id,
    "job_name": target_job_name,
    "job_id": job_id,
    "proposed_action": proposed_action,
    "actual_action": actual_action,
    "deployment_succeeded": deployment_succeeded,
    "deployment_status": deployment_status,
    "status_code": deploy_api_result.get("status_code"),
    "api_response_preview": deploy_api_result.get("response_text"),
    "task_count": len(job_tasks),
    "schedule_pause_status": lakeflow_job_payload.get("schedule", {}).get("pause_status"),
    "deploy_started_at": deploy_started_at.isoformat(),
    "deploy_finished_at": deploy_finished_at.isoformat(),
    "deploy_duration_seconds": (deploy_finished_at - deploy_started_at).total_seconds()
}

print("Lakeflow Jobs deployment result:")
print(json.dumps(deployment_result, indent=2))

# ─────────────────────────────────────────────────────────────
# Save deployment event to Delta
# ─────────────────────────────────────────────────────────────

DEPLOYMENT_EVENTS_TABLE = f"{GOLD_SCHEMA}.gold_workflow_job_deployment_events"

deployment_event_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("workflow_name", StringType(), True),
    StructField("workflow_version", StringType(), True),
    StructField("job_name", StringType(), True),
    StructField("job_id", StringType(), True),
    StructField("proposed_action", StringType(), True),
    StructField("actual_action", StringType(), True),
    StructField("deployment_succeeded", BooleanType(), True),
    StructField("deployment_status", StringType(), True),
    StructField("status_code", IntegerType(), True),
    StructField("task_count", IntegerType(), True),
    StructField("schedule_pause_status", StringType(), True),
    StructField("api_response_preview", StringType(), True),
    StructField("deployment_result_json", StringType(), True),
    StructField("deploy_started_at", TimestampType(), True),
    StructField("deploy_finished_at", TimestampType(), True),
])

deployment_event_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "workflow_name": WORKFLOW_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "job_name": target_job_name,
    "job_id": job_id,
    "proposed_action": proposed_action,
    "actual_action": actual_action,
    "deployment_succeeded": deployment_succeeded,
    "deployment_status": deployment_status,
    "status_code": int(deploy_api_result.get("status_code")) if deploy_api_result.get("status_code") is not None else None,
    "task_count": int(len(job_tasks)),
    "schedule_pause_status": lakeflow_job_payload.get("schedule", {}).get("pause_status"),
    "api_response_preview": str(deploy_api_result.get("response_text"))[:2000],
    "deployment_result_json": json.dumps(deployment_result, indent=2, default=str),
    "deploy_started_at": deploy_started_at,
    "deploy_finished_at": deploy_finished_at
}]

deployment_event_df = spark.createDataFrame(
    deployment_event_rows,
    schema=deployment_event_schema
)

(
    deployment_event_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(DEPLOYMENT_EVENTS_TABLE)
)

print(f"\nSaved deployment event to: {DEPLOYMENT_EVENTS_TABLE}")

# ─────────────────────────────────────────────────────────────
# Update latest job manifest row
# ─────────────────────────────────────────────────────────────

deployment_update_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "job_id": job_id,
    "job_created_or_updated": deployment_succeeded,
    "deployment_status": deployment_status,
    "deployment_updated_at": deploy_finished_at
}]

deployment_update_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("job_id", StringType(), True),
    StructField("job_created_or_updated", BooleanType(), True),
    StructField("deployment_status", StringType(), True),
    StructField("deployment_updated_at", TimestampType(), True),
])

deployment_update_df = spark.createDataFrame(
    deployment_update_rows,
    schema=deployment_update_schema
)

deployment_update_df.createOrReplaceTempView("workflow_deployment_update")

spark.sql(f"""
MERGE INTO {WORKFLOW_JOB_MANIFEST_TABLE} AS target
USING workflow_deployment_update AS source
ON target.workflow_deployment_id = source.workflow_deployment_id
WHEN MATCHED THEN UPDATE SET
  target.job_id = source.job_id,
  target.job_created_or_updated = source.job_created_or_updated,
  target.deployment_status = source.deployment_status
""")

print(f"Updated workflow manifest table: {WORKFLOW_JOB_MANIFEST_TABLE}")

display(
    spark.table(WORKFLOW_JOB_MANIFEST_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_name",
        "job_id",
        "task_count",
        "schedule_pause_status",
        "job_created_or_updated",
        "deployment_status",
        "created_at"
    )
)

if deployment_succeeded:
    print("\nLakeflow Job created/updated successfully.")
    print(f"Job ID: {job_id}")
    print("The job is created in PAUSED state by design.")
else:
    print("\nLakeflow Job deployment failed.")
    print("Review api_response_preview above.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 7
# Verify deployed Lakeflow Job and save final deployment summary
# ─────────────────────────────────────────────────────────────

import json
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    BooleanType,
    TimestampType
)

assert "deployment_result" in globals(), "deployment_result missing. Rerun Cell 6A."
assert "serverless_job_payload" in globals(), "serverless_job_payload missing. Rerun Cell 6A."
assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."

verify_started_at = datetime.now(timezone.utc)

JOB_ID = str(deployment_result.get("job_id"))

if JOB_ID is None or JOB_ID == "None":
    raise ValueError("JOB_ID is missing. Job deployment did not succeed.")

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

# ─────────────────────────────────────────────────────────────
# Fetch deployed job settings from Databricks Jobs API
# ─────────────────────────────────────────────────────────────

def get_databricks_job(job_id: str) -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/get"
    params = {"job_id": job_id}

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=60
    )

    if response.status_code != 200:
        return {
            "ok": False,
            "status_code": response.status_code,
            "response_text": response.text[:3000],
            "job": None
        }

    return {
        "ok": True,
        "status_code": response.status_code,
        "response_text": response.text[:3000],
        "job": response.json()
    }


job_get_result = get_databricks_job(JOB_ID)

if not job_get_result["ok"]:
    raise ValueError(
        f"Could not verify Databricks job. "
        f"HTTP {job_get_result['status_code']}: {job_get_result['response_text']}"
    )

deployed_job = job_get_result["job"]
deployed_settings = deployed_job.get("settings", {}) or {}

deployed_tasks = deployed_settings.get("tasks", []) or []
deployed_schedule = deployed_settings.get("schedule", {}) or {}
deployed_tags = deployed_settings.get("tags", {}) or {}

# ─────────────────────────────────────────────────────────────
# Product-readiness checks
# ─────────────────────────────────────────────────────────────

expected_task_count = len(serverless_job_payload.get("tasks", []))
actual_task_count = len(deployed_tasks)

tasks_with_classic_cluster = 0

for task in deployed_tasks:
    if (
        "job_cluster_key" in task
        or "new_cluster" in task
        or "existing_cluster_id" in task
    ):
        tasks_with_classic_cluster += 1

checks = {
    "job_exists": True,
    "job_name_matches": deployed_settings.get("name") == serverless_job_payload.get("name"),
    "task_count_matches": actual_task_count == expected_task_count,
    "schedule_exists": bool(deployed_schedule),
    "schedule_is_paused": deployed_schedule.get("pause_status") == "PAUSED",
    "timezone_matches": deployed_schedule.get("timezone_id") == JOB_TIMEZONE_ID,
    "cron_matches": deployed_schedule.get("quartz_cron_expression") == JOB_QUARTZ_CRON,
    "max_concurrent_runs_is_one": deployed_settings.get("max_concurrent_runs") == 1,
    "serverless_payload_confirmed": tasks_with_classic_cluster == 0,
    "project_tag_present": deployed_tags.get("project") == "supplysage_ai"
}

passed_checks = sum(1 for value in checks.values() if value is True)
failed_checks = sum(1 for value in checks.values() if value is not True)

overall_deployment_verified = failed_checks == 0

if overall_deployment_verified:
    final_product_status = "lakeflow_job_deployed_verified_paused"
else:
    final_product_status = "lakeflow_job_deployed_with_warnings"

verify_finished_at = datetime.now(timezone.utc)

job_url = f"{DATABRICKS_WORKSPACE_URL}/jobs/{JOB_ID}"

deployment_summary = {
    "workflow_deployment_id": workflow_deployment_id,
    "project_name": PROJECT_NAME,
    "workflow_name": WORKFLOW_NAME,
    "workflow_display_name": WORKFLOW_DISPLAY_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "job_id": JOB_ID,
    "job_name": deployed_settings.get("name"),
    "job_url": job_url,
    "expected_task_count": expected_task_count,
    "actual_task_count": actual_task_count,
    "schedule_quartz_cron": deployed_schedule.get("quartz_cron_expression"),
    "schedule_timezone_id": deployed_schedule.get("timezone_id"),
    "schedule_pause_status": deployed_schedule.get("pause_status"),
    "compute_mode": "serverless",
    "tasks_with_classic_cluster": tasks_with_classic_cluster,
    "max_concurrent_runs": deployed_settings.get("max_concurrent_runs"),
    "passed_checks": passed_checks,
    "failed_checks": failed_checks,
    "overall_deployment_verified": overall_deployment_verified,
    "final_product_status": final_product_status,
    "verification_checks": checks,
    "verified_at_utc": verify_finished_at.isoformat()
}

print("Lakeflow Job verification summary:")
print(json.dumps(deployment_summary, indent=2))

# ─────────────────────────────────────────────────────────────
# Save final deployment summary to Delta
# ─────────────────────────────────────────────────────────────

summary_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("project_name", StringType(), True),
    StructField("workflow_name", StringType(), True),
    StructField("workflow_display_name", StringType(), True),
    StructField("workflow_version", StringType(), True),
    StructField("job_id", StringType(), True),
    StructField("job_name", StringType(), True),
    StructField("job_url", StringType(), True),
    StructField("expected_task_count", IntegerType(), True),
    StructField("actual_task_count", IntegerType(), True),
    StructField("schedule_quartz_cron", StringType(), True),
    StructField("schedule_timezone_id", StringType(), True),
    StructField("schedule_pause_status", StringType(), True),
    StructField("compute_mode", StringType(), True),
    StructField("tasks_with_classic_cluster", IntegerType(), True),
    StructField("max_concurrent_runs", IntegerType(), True),
    StructField("passed_checks", IntegerType(), True),
    StructField("failed_checks", IntegerType(), True),
    StructField("overall_deployment_verified", BooleanType(), True),
    StructField("final_product_status", StringType(), True),
    StructField("verification_checks_json", StringType(), True),
    StructField("deployed_job_settings_json", StringType(), True),
    StructField("verified_at", TimestampType(), True),
])

summary_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "project_name": PROJECT_NAME,
    "workflow_name": WORKFLOW_NAME,
    "workflow_display_name": WORKFLOW_DISPLAY_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "job_id": JOB_ID,
    "job_name": deployed_settings.get("name"),
    "job_url": job_url,
    "expected_task_count": int(expected_task_count),
    "actual_task_count": int(actual_task_count),
    "schedule_quartz_cron": deployed_schedule.get("quartz_cron_expression"),
    "schedule_timezone_id": deployed_schedule.get("timezone_id"),
    "schedule_pause_status": deployed_schedule.get("pause_status"),
    "compute_mode": "serverless",
    "tasks_with_classic_cluster": int(tasks_with_classic_cluster),
    "max_concurrent_runs": int(deployed_settings.get("max_concurrent_runs") or 0),
    "passed_checks": int(passed_checks),
    "failed_checks": int(failed_checks),
    "overall_deployment_verified": bool(overall_deployment_verified),
    "final_product_status": final_product_status,
    "verification_checks_json": json.dumps(checks, indent=2, default=str),
    "deployed_job_settings_json": json.dumps(deployed_settings, indent=2, default=str),
    "verified_at": verify_finished_at
}]

deployment_summary_df = spark.createDataFrame(
    summary_rows,
    schema=summary_schema
)

(
    deployment_summary_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(WORKFLOW_DEPLOYMENT_SUMMARY_TABLE)
)

print(f"\nSaved final workflow deployment summary to: {WORKFLOW_DEPLOYMENT_SUMMARY_TABLE}")

# ─────────────────────────────────────────────────────────────
# Update main manifest status
# ─────────────────────────────────────────────────────────────

deployment_final_update_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("deployment_status", StringType(), True),
    StructField("job_id", StringType(), True),
    StructField("job_created_or_updated", BooleanType(), True),
])

deployment_final_update_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "deployment_status": final_product_status,
    "job_id": JOB_ID,
    "job_created_or_updated": True
}]

deployment_final_update_df = spark.createDataFrame(
    deployment_final_update_rows,
    schema=deployment_final_update_schema
)

deployment_final_update_df.createOrReplaceTempView("workflow_final_deployment_update")

spark.sql(f"""
MERGE INTO {WORKFLOW_JOB_MANIFEST_TABLE} AS target
USING workflow_final_deployment_update AS source
ON target.workflow_deployment_id = source.workflow_deployment_id
WHEN MATCHED THEN UPDATE SET
  target.deployment_status = source.deployment_status,
  target.job_id = source.job_id,
  target.job_created_or_updated = source.job_created_or_updated
""")

print(f"Updated workflow manifest final status: {WORKFLOW_JOB_MANIFEST_TABLE}")

display(
    spark.table(WORKFLOW_DEPLOYMENT_SUMMARY_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_id",
        "job_name",
        "expected_task_count",
        "actual_task_count",
        "schedule_pause_status",
        "compute_mode",
        "passed_checks",
        "failed_checks",
        "overall_deployment_verified",
        "final_product_status",
        "verified_at"
    )
    .orderBy(F.desc("verified_at"))
)

print("\nCell 7 complete.")

if overall_deployment_verified:
    print("Lakeflow deployment is verified and product-ready in PAUSED state.")
    print(f"Job URL: {job_url}")
else:
    print("Lakeflow deployment completed with warnings. Review failed checks above.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 8
# Unpause Lakeflow Job schedule and verify
# ─────────────────────────────────────────────────────────────

import json
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
    TimestampType
)

assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."
assert "workflow_deployment_id" in globals(), "workflow_deployment_id missing. Rerun Cell 1."
assert "JOB_QUARTZ_CRON" in globals(), "JOB_QUARTZ_CRON missing. Rerun Cell 1."
assert "JOB_TIMEZONE_ID" in globals(), "JOB_TIMEZONE_ID missing. Rerun Cell 1."

# Use JOB_ID from Cell 7 if available, otherwise use deployment_result.
if "JOB_ID" in globals() and JOB_ID:
    target_job_id = str(JOB_ID)
elif "deployment_result" in globals() and deployment_result.get("job_id"):
    target_job_id = str(deployment_result.get("job_id"))
else:
    raise ValueError("No job ID found. Rerun Cell 6A and Cell 7 first.")

UNPAUSE_JOB_NOW = True

if not UNPAUSE_JOB_NOW:
    raise ValueError("UNPAUSE_JOB_NOW is False. Set it to True only when ready to unpause the schedule.")

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

unpause_started_at = datetime.now(timezone.utc)

# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────

def get_databricks_job(job_id: str) -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/get"
    params = {"job_id": job_id}

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=60
    )

    if response.status_code != 200:
        return {
            "ok": False,
            "status_code": response.status_code,
            "response_text": response.text[:3000],
            "job": None
        }

    return {
        "ok": True,
        "status_code": response.status_code,
        "response_text": response.text[:3000],
        "job": response.json()
    }


def update_job_schedule_pause_status(job_id: str, pause_status: str) -> dict:
    """
    Try Jobs API 2.2 first, then fall back to 2.1 if needed.
    """
    update_payload = {
        "job_id": int(job_id),
        "new_settings": {
            "schedule": {
                "quartz_cron_expression": JOB_QUARTZ_CRON,
                "timezone_id": JOB_TIMEZONE_ID,
                "pause_status": pause_status
            }
        }
    }

    api_versions = ["2.2", "2.1"]

    last_response = None

    for api_version in api_versions:
        url = f"{DATABRICKS_WORKSPACE_URL}/api/{api_version}/jobs/update"

        response = requests.post(
            url,
            headers=headers,
            data=json.dumps(update_payload),
            timeout=120
        )

        last_response = response

        if response.status_code in [200, 201]:
            return {
                "ok": True,
                "api_version": api_version,
                "status_code": response.status_code,
                "response_text": response.text[:3000]
            }

        # If endpoint/version unsupported, try fallback version.
        if response.status_code in [404, 405]:
            continue

        # Other errors are real errors.
        break

    return {
        "ok": False,
        "api_version": None,
        "status_code": last_response.status_code if last_response is not None else None,
        "response_text": last_response.text[:3000] if last_response is not None else "No response"
    }


# ─────────────────────────────────────────────────────────────
# Before state
# ─────────────────────────────────────────────────────────────

before_get = get_databricks_job(target_job_id)

if not before_get["ok"]:
    raise ValueError(
        f"Could not fetch job before unpause. "
        f"HTTP {before_get['status_code']}: {before_get['response_text']}"
    )

before_job = before_get["job"]
before_settings = before_job.get("settings", {}) or {}
before_schedule = before_settings.get("schedule", {}) or {}

print("Before unpause:")
print(json.dumps({
    "job_id": target_job_id,
    "job_name": before_settings.get("name"),
    "pause_status": before_schedule.get("pause_status"),
    "quartz_cron_expression": before_schedule.get("quartz_cron_expression"),
    "timezone_id": before_schedule.get("timezone_id")
}, indent=2))

# ─────────────────────────────────────────────────────────────
# Unpause schedule
# ─────────────────────────────────────────────────────────────

unpause_api_result = update_job_schedule_pause_status(
    job_id=target_job_id,
    pause_status="UNPAUSED"
)

unpause_finished_at = datetime.now(timezone.utc)

print("\nUnpause API result:")
print(json.dumps(unpause_api_result, indent=2))

if not unpause_api_result["ok"]:
    raise ValueError(
        f"Failed to unpause job schedule. "
        f"HTTP {unpause_api_result['status_code']}: {unpause_api_result['response_text']}"
    )

# ─────────────────────────────────────────────────────────────
# Verify after state
# ─────────────────────────────────────────────────────────────

after_get = get_databricks_job(target_job_id)

if not after_get["ok"]:
    raise ValueError(
        f"Could not fetch job after unpause. "
        f"HTTP {after_get['status_code']}: {after_get['response_text']}"
    )

after_job = after_get["job"]
after_settings = after_job.get("settings", {}) or {}
after_schedule = after_settings.get("schedule", {}) or {}

schedule_unpaused = after_schedule.get("pause_status") == "UNPAUSED"
cron_matches = after_schedule.get("quartz_cron_expression") == JOB_QUARTZ_CRON
timezone_matches = after_schedule.get("timezone_id") == JOB_TIMEZONE_ID

unpause_verified = bool(schedule_unpaused and cron_matches and timezone_matches)

if unpause_verified:
    final_schedule_status = "lakeflow_job_schedule_unpaused_verified"
else:
    final_schedule_status = "lakeflow_job_schedule_unpaused_with_warnings"

unpause_result = {
    "workflow_deployment_id": workflow_deployment_id,
    "job_id": target_job_id,
    "job_name": after_settings.get("name"),
    "job_url": f"{DATABRICKS_WORKSPACE_URL}/jobs/{target_job_id}",
    "before_pause_status": before_schedule.get("pause_status"),
    "after_pause_status": after_schedule.get("pause_status"),
    "schedule_quartz_cron": after_schedule.get("quartz_cron_expression"),
    "schedule_timezone_id": after_schedule.get("timezone_id"),
    "schedule_unpaused": schedule_unpaused,
    "cron_matches": cron_matches,
    "timezone_matches": timezone_matches,
    "unpause_verified": unpause_verified,
    "final_schedule_status": final_schedule_status,
    "api_version_used": unpause_api_result.get("api_version"),
    "status_code": unpause_api_result.get("status_code"),
    "unpause_started_at": unpause_started_at.isoformat(),
    "unpause_finished_at": unpause_finished_at.isoformat(),
    "unpause_duration_seconds": (unpause_finished_at - unpause_started_at).total_seconds()
}

print("\nAfter unpause verification:")
print(json.dumps(unpause_result, indent=2))

# ─────────────────────────────────────────────────────────────
# Save unpause event
# ─────────────────────────────────────────────────────────────

UNPAUSE_EVENTS_TABLE = f"{GOLD_SCHEMA}.gold_workflow_job_schedule_events"

unpause_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("workflow_name", StringType(), True),
    StructField("workflow_version", StringType(), True),
    StructField("job_id", StringType(), True),
    StructField("job_name", StringType(), True),
    StructField("job_url", StringType(), True),
    StructField("before_pause_status", StringType(), True),
    StructField("after_pause_status", StringType(), True),
    StructField("schedule_quartz_cron", StringType(), True),
    StructField("schedule_timezone_id", StringType(), True),
    StructField("schedule_unpaused", BooleanType(), True),
    StructField("cron_matches", BooleanType(), True),
    StructField("timezone_matches", BooleanType(), True),
    StructField("unpause_verified", BooleanType(), True),
    StructField("final_schedule_status", StringType(), True),
    StructField("api_version_used", StringType(), True),
    StructField("status_code", IntegerType(), True),
    StructField("unpause_result_json", StringType(), True),
    StructField("unpause_started_at", TimestampType(), True),
    StructField("unpause_finished_at", TimestampType(), True),
])

unpause_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "workflow_name": WORKFLOW_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "job_id": target_job_id,
    "job_name": after_settings.get("name"),
    "job_url": f"{DATABRICKS_WORKSPACE_URL}/jobs/{target_job_id}",
    "before_pause_status": before_schedule.get("pause_status"),
    "after_pause_status": after_schedule.get("pause_status"),
    "schedule_quartz_cron": after_schedule.get("quartz_cron_expression"),
    "schedule_timezone_id": after_schedule.get("timezone_id"),
    "schedule_unpaused": bool(schedule_unpaused),
    "cron_matches": bool(cron_matches),
    "timezone_matches": bool(timezone_matches),
    "unpause_verified": bool(unpause_verified),
    "final_schedule_status": final_schedule_status,
    "api_version_used": unpause_api_result.get("api_version"),
    "status_code": int(unpause_api_result.get("status_code")) if unpause_api_result.get("status_code") is not None else None,
    "unpause_result_json": json.dumps(unpause_result, indent=2, default=str),
    "unpause_started_at": unpause_started_at,
    "unpause_finished_at": unpause_finished_at
}]

unpause_df = spark.createDataFrame(unpause_rows, schema=unpause_schema)

(
    unpause_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(UNPAUSE_EVENTS_TABLE)
)

print(f"\nSaved schedule event to: {UNPAUSE_EVENTS_TABLE}")

# ─────────────────────────────────────────────────────────────
# Update deployment summary and manifest final status
# ─────────────────────────────────────────────────────────────

schedule_update_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("schedule_pause_status", StringType(), True),
    StructField("final_product_status", StringType(), True),
])

schedule_update_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "schedule_pause_status": after_schedule.get("pause_status"),
    "final_product_status": final_schedule_status
}]

schedule_update_df = spark.createDataFrame(
    schedule_update_rows,
    schema=schedule_update_schema
)

schedule_update_df.createOrReplaceTempView("workflow_schedule_update")

spark.sql(f"""
MERGE INTO {WORKFLOW_DEPLOYMENT_SUMMARY_TABLE} AS target
USING workflow_schedule_update AS source
ON target.workflow_deployment_id = source.workflow_deployment_id
WHEN MATCHED THEN UPDATE SET
  target.schedule_pause_status = source.schedule_pause_status,
  target.final_product_status = source.final_product_status
""")

spark.sql(f"""
MERGE INTO {WORKFLOW_JOB_MANIFEST_TABLE} AS target
USING workflow_schedule_update AS source
ON target.workflow_deployment_id = source.workflow_deployment_id
WHEN MATCHED THEN UPDATE SET
  target.schedule_pause_status = source.schedule_pause_status,
  target.deployment_status = source.final_product_status
""")

print(f"Updated deployment summary: {WORKFLOW_DEPLOYMENT_SUMMARY_TABLE}")
print(f"Updated job manifest: {WORKFLOW_JOB_MANIFEST_TABLE}")

display(
    spark.table(UNPAUSE_EVENTS_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_id",
        "job_name",
        "before_pause_status",
        "after_pause_status",
        "schedule_quartz_cron",
        "schedule_timezone_id",
        "unpause_verified",
        "final_schedule_status",
        "unpause_finished_at"
    )
    .orderBy(F.desc("unpause_finished_at"))
)

if unpause_verified:
    print("\nLakeflow Job schedule is now UNPAUSED and verified.")
    print("SupplySage AI is scheduled to run daily at 6:00 AM America/Chicago.")
else:
    print("\nSchedule update completed with warnings. Review verification output above.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 9A
# Trigger manual Lakeflow Job run with <=64 char idempotency token
# ─────────────────────────────────────────────────────────────

import json
import uuid
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
    LongType,
    TimestampType
)

assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."
assert "workflow_deployment_id" in globals(), "workflow_deployment_id missing. Rerun Cell 1."

if "JOB_ID" in globals() and JOB_ID:
    target_job_id = str(JOB_ID)
elif "deployment_result" in globals() and deployment_result.get("job_id"):
    target_job_id = str(deployment_result.get("job_id"))
else:
    raise ValueError("No job ID found. Rerun Cell 6A and Cell 7 first.")

TRIGGER_MANUAL_RUN_NOW = True

if not TRIGGER_MANUAL_RUN_NOW:
    raise ValueError("TRIGGER_MANUAL_RUN_NOW is False.")

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

manual_run_started_at = datetime.now(timezone.utc)

# Must be <=64 chars.
idempotency_token = f"ss_{manual_run_started_at.strftime('%Y%m%d%H%M%S')}_{uuid.uuid4().hex[:8]}"

print("Idempotency token:", idempotency_token)
print("Token length:", len(idempotency_token))

assert len(idempotency_token) <= 64, "Idempotency token is too long."

# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────

def trigger_job_run_now(job_id: str) -> dict:
    payload = {
        "job_id": int(job_id),
        "idempotency_token": idempotency_token
    }

    for api_version in ["2.2", "2.1"]:
        url = f"{DATABRICKS_WORKSPACE_URL}/api/{api_version}/jobs/run-now"

        response = requests.post(
            url,
            headers=headers,
            data=json.dumps(payload),
            timeout=120
        )

        if response.status_code in [200, 201]:
            result = response.json()

            return {
                "ok": True,
                "api_version": api_version,
                "status_code": response.status_code,
                "response_text": json.dumps(result, indent=2),
                "run_id": str(result.get("run_id")) if result.get("run_id") is not None else None,
                "number_in_job": int(result.get("number_in_job")) if result.get("number_in_job") is not None else None
            }

        if response.status_code in [404, 405]:
            continue

        return {
            "ok": False,
            "api_version": api_version,
            "status_code": response.status_code,
            "response_text": response.text[:3000],
            "run_id": None,
            "number_in_job": None
        }

    return {
        "ok": False,
        "api_version": None,
        "status_code": None,
        "response_text": "No supported Jobs API version succeeded.",
        "run_id": None,
        "number_in_job": None
    }


def get_job_run(run_id: str) -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/runs/get"
    params = {"run_id": run_id}

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=60
    )

    if response.status_code != 200:
        return {
            "ok": False,
            "status_code": response.status_code,
            "response_text": response.text[:3000],
            "run": None
        }

    return {
        "ok": True,
        "status_code": response.status_code,
        "response_text": response.text[:3000],
        "run": response.json()
    }


# ─────────────────────────────────────────────────────────────
# Trigger run
# ─────────────────────────────────────────────────────────────

run_now_result = trigger_job_run_now(target_job_id)
manual_run_triggered_at = datetime.now(timezone.utc)

print("\nManual Lakeflow run trigger result:")
print(json.dumps(run_now_result, indent=2))

if not run_now_result["ok"]:
    raise ValueError(
        f"Manual run trigger failed. "
        f"HTTP {run_now_result['status_code']}: {run_now_result['response_text']}"
    )

RUN_ID = run_now_result["run_id"]

if RUN_ID is None:
    raise ValueError("Run was triggered but no run_id was returned.")

# ─────────────────────────────────────────────────────────────
# Fetch initial run state
# ─────────────────────────────────────────────────────────────

run_get_result = get_job_run(RUN_ID)

if not run_get_result["ok"]:
    raise ValueError(
        f"Could not fetch run status. "
        f"HTTP {run_get_result['status_code']}: {run_get_result['response_text']}"
    )

run_payload = run_get_result["run"]
run_state = run_payload.get("state", {}) or {}

life_cycle_state = run_state.get("life_cycle_state")
result_state = run_state.get("result_state")
state_message = run_state.get("state_message")

run_page_url = (
    run_payload.get("run_page_url")
    or f"{DATABRICKS_WORKSPACE_URL}/jobs/{target_job_id}/runs/{RUN_ID}"
)

manual_run_result = {
    "workflow_deployment_id": workflow_deployment_id,
    "job_id": target_job_id,
    "run_id": RUN_ID,
    "number_in_job": run_now_result.get("number_in_job"),
    "run_page_url": run_page_url,
    "idempotency_token": idempotency_token,
    "idempotency_token_length": len(idempotency_token),
    "trigger_api_version": run_now_result.get("api_version"),
    "trigger_status_code": run_now_result.get("status_code"),
    "trigger_succeeded": bool(run_now_result.get("ok")),
    "initial_life_cycle_state": life_cycle_state,
    "initial_result_state": result_state,
    "initial_state_message": state_message,
    "manual_run_started_at": manual_run_started_at.isoformat(),
    "manual_run_triggered_at": manual_run_triggered_at.isoformat()
}

print("\nInitial manual run status:")
print(json.dumps(manual_run_result, indent=2))

# ─────────────────────────────────────────────────────────────
# Save manual run event
# ─────────────────────────────────────────────────────────────

MANUAL_RUN_EVENTS_TABLE = f"{GOLD_SCHEMA}.gold_workflow_job_manual_run_events"

manual_run_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("workflow_name", StringType(), True),
    StructField("workflow_version", StringType(), True),
    StructField("job_id", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("number_in_job", LongType(), True),
    StructField("run_page_url", StringType(), True),
    StructField("idempotency_token", StringType(), True),
    StructField("trigger_api_version", StringType(), True),
    StructField("trigger_status_code", IntegerType(), True),
    StructField("trigger_succeeded", BooleanType(), True),
    StructField("initial_life_cycle_state", StringType(), True),
    StructField("initial_result_state", StringType(), True),
    StructField("initial_state_message", StringType(), True),
    StructField("manual_run_result_json", StringType(), True),
    StructField("manual_run_started_at", TimestampType(), True),
    StructField("manual_run_triggered_at", TimestampType(), True),
])

manual_run_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "workflow_name": WORKFLOW_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "job_id": target_job_id,
    "run_id": RUN_ID,
    "number_in_job": run_now_result.get("number_in_job"),
    "run_page_url": run_page_url,
    "idempotency_token": idempotency_token,
    "trigger_api_version": run_now_result.get("api_version"),
    "trigger_status_code": int(run_now_result.get("status_code")) if run_now_result.get("status_code") is not None else None,
    "trigger_succeeded": bool(run_now_result.get("ok")),
    "initial_life_cycle_state": life_cycle_state,
    "initial_result_state": result_state,
    "initial_state_message": state_message,
    "manual_run_result_json": json.dumps(manual_run_result, indent=2, default=str),
    "manual_run_started_at": manual_run_started_at,
    "manual_run_triggered_at": manual_run_triggered_at
}]

manual_run_df = spark.createDataFrame(
    manual_run_rows,
    schema=manual_run_schema
)

(
    manual_run_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(MANUAL_RUN_EVENTS_TABLE)
)

print(f"\nSaved manual run event to: {MANUAL_RUN_EVENTS_TABLE}")

display(
    spark.table(MANUAL_RUN_EVENTS_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_id",
        "run_id",
        "trigger_succeeded",
        "initial_life_cycle_state",
        "initial_result_state",
        "initial_state_message",
        "run_page_url",
        "manual_run_triggered_at"
    )
    .orderBy(F.desc("manual_run_triggered_at"))
)

print("\nManual Lakeflow Job run has been triggered.")
print(f"Run ID: {RUN_ID}")
print(f"Run URL: {run_page_url}")
print("Watch the Databricks Jobs UI for all 26 tasks.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 10
# Check latest manual Lakeflow Job run status
# ─────────────────────────────────────────────────────────────

import json
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
    TimestampType
)

assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."

if "RUN_ID" in globals() and RUN_ID:
    target_run_id = str(RUN_ID)
else:
    # Fallback to latest manual run event saved in Delta
    latest_run_row = (
        spark.table(f"{GOLD_SCHEMA}.gold_workflow_job_manual_run_events")
        .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
        .orderBy(F.col("manual_run_triggered_at").desc())
        .select("run_id")
        .first()
    )

    if latest_run_row is None:
        raise ValueError("No RUN_ID found. Trigger a manual run first.")

    target_run_id = str(latest_run_row["run_id"])

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

status_checked_at = datetime.now(timezone.utc)

def get_job_run(run_id: str) -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/runs/get"
    params = {"run_id": run_id}

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=60
    )

    if response.status_code != 200:
        return {
            "ok": False,
            "status_code": response.status_code,
            "response_text": response.text[:3000],
            "run": None
        }

    return {
        "ok": True,
        "status_code": response.status_code,
        "response_text": response.text[:3000],
        "run": response.json()
    }


run_get_result = get_job_run(target_run_id)

if not run_get_result["ok"]:
    raise ValueError(
        f"Could not fetch run status. "
        f"HTTP {run_get_result['status_code']}: {run_get_result['response_text']}"
    )

run_payload = run_get_result["run"]
run_state = run_payload.get("state", {}) or {}
tasks = run_payload.get("tasks", []) or []

life_cycle_state = run_state.get("life_cycle_state")
result_state = run_state.get("result_state")
state_message = run_state.get("state_message")

run_page_url = run_payload.get("run_page_url")
job_id = str(run_payload.get("job_id")) if run_payload.get("job_id") is not None else None

task_status_rows = []

for task in tasks:
    task_state = task.get("state", {}) or {}

    task_status_rows.append({
        "run_id": target_run_id,
        "job_id": job_id,
        "task_key": task.get("task_key"),
        "run_page_url": task.get("run_page_url"),
        "life_cycle_state": task_state.get("life_cycle_state"),
        "result_state": task_state.get("result_state"),
        "state_message": task_state.get("state_message"),
        "start_time": task.get("start_time"),
        "end_time": task.get("end_time"),
        "status_checked_at": status_checked_at
    })

total_tasks = len(task_status_rows)
running_tasks = sum(1 for row in task_status_rows if row["life_cycle_state"] == "RUNNING")
pending_tasks = sum(1 for row in task_status_rows if row["life_cycle_state"] == "PENDING")
terminated_tasks = sum(1 for row in task_status_rows if row["life_cycle_state"] == "TERMINATED")
skipped_tasks = sum(1 for row in task_status_rows if row["life_cycle_state"] == "SKIPPED")
failed_tasks = sum(1 for row in task_status_rows if row["result_state"] in ["FAILED", "TIMEDOUT", "CANCELED"])

run_status_summary = {
    "workflow_deployment_id": workflow_deployment_id,
    "job_id": job_id,
    "run_id": target_run_id,
    "run_page_url": run_page_url,
    "life_cycle_state": life_cycle_state,
    "result_state": result_state,
    "state_message": state_message,
    "total_tasks": total_tasks,
    "running_tasks": running_tasks,
    "pending_tasks": pending_tasks,
    "terminated_tasks": terminated_tasks,
    "skipped_tasks": skipped_tasks,
    "failed_tasks": failed_tasks,
    "status_checked_at": status_checked_at.isoformat()
}

print("Lakeflow manual run status summary:")
print(json.dumps(run_status_summary, indent=2))

if len(task_status_rows) > 0:
    task_status_schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("job_id", StringType(), True),
        StructField("task_key", StringType(), True),
        StructField("run_page_url", StringType(), True),
        StructField("life_cycle_state", StringType(), True),
        StructField("result_state", StringType(), True),
        StructField("state_message", StringType(), True),
        StructField("start_time", StringType(), True),
        StructField("end_time", StringType(), True),
        StructField("status_checked_at", TimestampType(), True),
    ])

    task_status_df = spark.createDataFrame(
        task_status_rows,
        schema=task_status_schema
    )

    display(
        task_status_df.select(
            "task_key",
            "life_cycle_state",
            "result_state",
            "state_message",
            "run_page_url"
        )
    )

# Save run status snapshot
RUN_STATUS_TABLE = f"{GOLD_SCHEMA}.gold_workflow_job_run_status_snapshots"

run_status_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("job_id", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("run_page_url", StringType(), True),
    StructField("life_cycle_state", StringType(), True),
    StructField("result_state", StringType(), True),
    StructField("state_message", StringType(), True),
    StructField("total_tasks", IntegerType(), True),
    StructField("running_tasks", IntegerType(), True),
    StructField("pending_tasks", IntegerType(), True),
    StructField("terminated_tasks", IntegerType(), True),
    StructField("skipped_tasks", IntegerType(), True),
    StructField("failed_tasks", IntegerType(), True),
    StructField("run_status_summary_json", StringType(), True),
    StructField("status_checked_at", TimestampType(), True),
])

run_status_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "job_id": job_id,
    "run_id": target_run_id,
    "run_page_url": run_page_url,
    "life_cycle_state": life_cycle_state,
    "result_state": result_state,
    "state_message": state_message,
    "total_tasks": int(total_tasks),
    "running_tasks": int(running_tasks),
    "pending_tasks": int(pending_tasks),
    "terminated_tasks": int(terminated_tasks),
    "skipped_tasks": int(skipped_tasks),
    "failed_tasks": int(failed_tasks),
    "run_status_summary_json": json.dumps(run_status_summary, indent=2, default=str),
    "status_checked_at": status_checked_at
}]

run_status_df = spark.createDataFrame(
    run_status_rows,
    schema=run_status_schema
)

(
    run_status_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(RUN_STATUS_TABLE)
)

print(f"\nSaved run status snapshot to: {RUN_STATUS_TABLE}")

if life_cycle_state == "TERMINATED" and result_state == "SUCCESS":
    print("\nFull Lakeflow run completed successfully.")
elif failed_tasks > 0 or result_state in ["FAILED", "TIMEDOUT", "CANCELED"]:
    print("\nLakeflow run has failures. Check failed task details in the Jobs UI.")
else:
    print("\nLakeflow run is still in progress.")
    print(f"Run URL: {run_page_url}")

In [0]:
# ─────────────────────────────────────────────────────────────
# Find currently active runs for the SupplySage Lakeflow job
# ─────────────────────────────────────────────────────────────

import requests
import json
from datetime import datetime, timezone

JOB_ID = 522140299855346

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_WORKSPACE_URL = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

runs_list_url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/runs/list"

response = requests.get(
    runs_list_url,
    headers=headers,
    params={
        "job_id": JOB_ID,
        "active_only": "true",
        "limit": 10
    },
    timeout=60
)

print("Status code:", response.status_code)
response.raise_for_status()

runs_info = response.json()
active_runs = runs_info.get("runs", [])

print("Active runs found:", len(active_runs))

for run in active_runs:
    run_id = run.get("run_id")
    number_in_job = run.get("number_in_job")
    state = run.get("state", {})
    start_time = run.get("start_time")
    run_page_url = run.get("run_page_url")

    print("\nRun ID:", run_id)
    print("Number in job:", number_in_job)
    print("Life cycle:", state.get("life_cycle_state"))
    print("Result:", state.get("result_state"))
    print("Message:", state.get("state_message"))
    print("Start time ms:", start_time)
    print("Run URL:", run_page_url)

print("\nChecked at UTC:", datetime.now(timezone.utc).isoformat())

In [0]:
# ─────────────────────────────────────────────────────────────
# Inspect active Lakeflow run task states
# ─────────────────────────────────────────────────────────────

import requests
import json
from datetime import datetime, timezone

RUN_ID = 7402120575079

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_WORKSPACE_URL = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

run_get_url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/runs/get"

response = requests.get(
    run_get_url,
    headers=headers,
    params={"run_id": RUN_ID},
    timeout=60
)

print("Status code:", response.status_code)
response.raise_for_status()

run_info = response.json()

state = run_info.get("state", {})
print("\nRun state:")
print(json.dumps(state, indent=2))

tasks = run_info.get("tasks", [])

print("\nTask states:")
for task in tasks:
    task_key = task.get("task_key")
    task_state = task.get("state", {})
    life_cycle = task_state.get("life_cycle_state")
    result = task_state.get("result_state")
    state_message = task_state.get("state_message")

    print(f"{task_key} | life_cycle={life_cycle} | result={result} | message={state_message}")

active_tasks = [
    task for task in tasks
    if task.get("state", {}).get("life_cycle_state") in ["PENDING", "RUNNING", "TERMINATING"]
]

failed_tasks = [
    task for task in tasks
    if task.get("state", {}).get("result_state") in ["FAILED", "TIMEDOUT", "CANCELED"]
]

print("\nCurrently active / pending tasks:")
for task in active_tasks:
    print(task.get("task_key"), task.get("state", {}))

print("\nFailed tasks:")
for task in failed_tasks:
    print(task.get("task_key"), task.get("state", {}))

print("\nChecked at UTC:", datetime.now(timezone.utc).isoformat())

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 10A
# Inspect failed Lakeflow task details + notebook output
# ─────────────────────────────────────────────────────────────

import json
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType
)

assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."
assert "GOLD_SCHEMA" in globals(), "GOLD_SCHEMA missing. Rerun Cell 1."

# Use active RUN_ID if available, otherwise latest manual run event.
if "RUN_ID" in globals() and RUN_ID:
    target_run_id = str(RUN_ID)
else:
    latest_run_row = (
        spark.table(f"{GOLD_SCHEMA}.gold_workflow_job_manual_run_events")
        .orderBy(F.col("manual_run_triggered_at").desc())
        .select("run_id")
        .first()
    )

    if latest_run_row is None:
        raise ValueError("No RUN_ID found. Trigger or locate a manual run first.")

    target_run_id = str(latest_run_row["run_id"])

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

checked_at = datetime.now(timezone.utc)

def api_get(path: str, params=None, api_version="2.1") -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/{api_version}/{path.lstrip('/')}"
    response = requests.get(
        url,
        headers=headers,
        params=params or {},
        timeout=90
    )

    parsed_json = {}
    try:
        parsed_json = response.json()
    except Exception:
        parsed_json = {}

    return {
        "ok": response.status_code == 200,
        "status_code": response.status_code,
        "text": response.text[:12000],
        "json": parsed_json
    }


# ─────────────────────────────────────────────────────────────
# 1. Fetch parent run
# ─────────────────────────────────────────────────────────────

run_result = api_get(
    path="jobs/runs/get",
    params={"run_id": int(target_run_id)},
    api_version="2.1"
)

if not run_result["ok"]:
    raise ValueError(
        f"Could not fetch parent run. "
        f"HTTP {run_result['status_code']}: {run_result['text']}"
    )

run_payload = run_result["json"]
run_state = run_payload.get("state", {}) or {}
tasks = run_payload.get("tasks", []) or []

parent_summary = {
    "run_id": target_run_id,
    "job_id": run_payload.get("job_id"),
    "run_page_url": run_payload.get("run_page_url"),
    "life_cycle_state": run_state.get("life_cycle_state"),
    "result_state": run_state.get("result_state"),
    "state_message": run_state.get("state_message"),
    "task_count": len(tasks),
    "checked_at": checked_at.isoformat()
}

print("Parent run summary:")
print(json.dumps(parent_summary, indent=2))

# ─────────────────────────────────────────────────────────────
# 2. Build task status detail table
# ─────────────────────────────────────────────────────────────

task_rows = []

for task in tasks:
    task_state = task.get("state", {}) or {}

    task_rows.append({
        "parent_run_id": target_run_id,
        "job_id": str(run_payload.get("job_id")) if run_payload.get("job_id") is not None else None,
        "task_key": task.get("task_key"),
        "task_run_id": str(task.get("run_id")) if task.get("run_id") is not None else None,
        "life_cycle_state": task_state.get("life_cycle_state"),
        "result_state": task_state.get("result_state"),
        "state_message": task_state.get("state_message"),
        "run_page_url": task.get("run_page_url"),
        "notebook_path": (
            (task.get("notebook_task") or {}).get("notebook_path")
            if task.get("notebook_task") else None
        ),
        "start_time": str(task.get("start_time")) if task.get("start_time") is not None else None,
        "end_time": str(task.get("end_time")) if task.get("end_time") is not None else None,
        "checked_at": checked_at
    })

task_schema = StructType([
    StructField("parent_run_id", StringType(), True),
    StructField("job_id", StringType(), True),
    StructField("task_key", StringType(), True),
    StructField("task_run_id", StringType(), True),
    StructField("life_cycle_state", StringType(), True),
    StructField("result_state", StringType(), True),
    StructField("state_message", StringType(), True),
    StructField("run_page_url", StringType(), True),
    StructField("notebook_path", StringType(), True),
    StructField("start_time", StringType(), True),
    StructField("end_time", StringType(), True),
    StructField("checked_at", TimestampType(), True),
])

task_status_df = spark.createDataFrame(task_rows, schema=task_schema)

print("\nTask status table:")
display(
    task_status_df
    .select(
        "task_key",
        "task_run_id",
        "life_cycle_state",
        "result_state",
        "state_message",
        "notebook_path",
        "run_page_url"
    )
    .orderBy(
        F.when(F.col("result_state").isin("FAILED", "TIMEDOUT", "CANCELED"), 0)
        .when(F.col("life_cycle_state") == "INTERNAL_ERROR", 1)
        .when(F.col("life_cycle_state") == "TERMINATED", 2)
        .otherwise(3),
        "task_key"
    )
)

failed_task_rows = [
    row for row in task_rows
    if row["result_state"] in ["FAILED", "TIMEDOUT", "CANCELED"]
    or row["life_cycle_state"] in ["INTERNAL_ERROR"]
]

print("\nFailed task count:", len(failed_task_rows))

if len(failed_task_rows) == 0:
    print("No failed task rows found in task array. Parent run may still be finalizing.")
else:
    print("\nFailed task keys:")
    print(json.dumps(
        [
            {
                "task_key": row["task_key"],
                "task_run_id": row["task_run_id"],
                "result_state": row["result_state"],
                "state_message": row["state_message"],
                "run_page_url": row["run_page_url"]
            }
            for row in failed_task_rows
        ],
        indent=2
    ))

# ─────────────────────────────────────────────────────────────
# 3. Fetch notebook output/error for each failed task
# ─────────────────────────────────────────────────────────────

failed_output_rows = []

for row in failed_task_rows:
    task_run_id = row["task_run_id"]

    if not task_run_id:
        continue

    output_result = api_get(
        path="jobs/runs/get-output",
        params={"run_id": int(task_run_id)},
        api_version="2.1"
    )

    output_json = output_result["json"] or {}

    notebook_output = output_json.get("notebook_output", {}) or {}
    error_text = output_json.get("error") or ""
    error_trace = output_json.get("error_trace") or ""

    result_text = notebook_output.get("result") or ""

    failed_output_rows.append({
        "parent_run_id": target_run_id,
        "task_key": row["task_key"],
        "task_run_id": task_run_id,
        "notebook_path": row["notebook_path"],
        "run_page_url": row["run_page_url"],
        "output_status_code": int(output_result["status_code"]) if output_result["status_code"] is not None else None,
        "state_message": row["state_message"],
        "notebook_result_preview": result_text[:5000],
        "error_preview": error_text[:5000],
        "error_trace_preview": error_trace[:8000],
        "raw_output_preview": output_result["text"][:8000],
        "checked_at": checked_at
    })

failed_output_schema = StructType([
    StructField("parent_run_id", StringType(), True),
    StructField("task_key", StringType(), True),
    StructField("task_run_id", StringType(), True),
    StructField("notebook_path", StringType(), True),
    StructField("run_page_url", StringType(), True),
    StructField("output_status_code", IntegerType(), True),
    StructField("state_message", StringType(), True),
    StructField("notebook_result_preview", StringType(), True),
    StructField("error_preview", StringType(), True),
    StructField("error_trace_preview", StringType(), True),
    StructField("raw_output_preview", StringType(), True),
    StructField("checked_at", TimestampType(), True),
])

failed_output_df = spark.createDataFrame(failed_output_rows, schema=failed_output_schema)

print("\nFailed task output/error preview:")
display(
    failed_output_df.select(
        "task_key",
        "task_run_id",
        "notebook_path",
        "state_message",
        "error_preview",
        "error_trace_preview",
        "run_page_url"
    )
)

# ─────────────────────────────────────────────────────────────
# 4. Save debug snapshot
# ─────────────────────────────────────────────────────────────

RUN_TASK_DEBUG_TABLE = f"{GOLD_SCHEMA}.gold_workflow_failed_task_debug"

(
    failed_output_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(RUN_TASK_DEBUG_TABLE)
)

print(f"\nSaved failed task debug snapshot to: {RUN_TASK_DEBUG_TABLE}")

print("\nCell 10A complete.")
print("Send me the failed task_key and error_preview/error_trace_preview output.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 11
# Add Notebook 32 alerting task to deployed Lakeflow Job
# ─────────────────────────────────────────────────────────────

import json
import copy
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
    TimestampType
)

assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."
assert "workflow_deployment_id" in globals(), "workflow_deployment_id missing. Rerun Cell 1."
assert "NOTEBOOK_BASE_PATH" in globals(), "NOTEBOOK_BASE_PATH missing. Rerun Cell 1."
assert "GOLD_SCHEMA" in globals(), "GOLD_SCHEMA missing. Rerun Cell 1."

# Existing deployed job
if "JOB_ID" in globals() and JOB_ID:
    target_job_id = str(JOB_ID)
elif "deployment_result" in globals() and deployment_result.get("job_id"):
    target_job_id = str(deployment_result.get("job_id"))
else:
    target_job_id = "522140299855346"

ALERTING_NOTEBOOK_TASK_KEY = "alerting_notification_routing"
ALERTING_NOTEBOOK_PATH = f"{NOTEBOOK_BASE_PATH}/32_alerting_notification_routing"

# We want alerting after serving health, before the workflow summary.
ALERTING_DEPENDS_ON = ["serving_health_check"]
FINAL_SUMMARY_TASK_KEY = "workflow_summary"

UPDATE_JOB_WITH_ALERTING_TASK = True

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

patch_started_at = datetime.now(timezone.utc)

# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────

def api_get(path: str, params=None, api_version="2.1") -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/{api_version}/{path.lstrip('/')}"
    response = requests.get(url, headers=headers, params=params or {}, timeout=90)

    return {
        "ok": response.status_code == 200,
        "status_code": response.status_code,
        "text": response.text[:5000],
        "json": response.json() if response.status_code == 200 and response.text else {}
    }


def api_post(path: str, payload: dict, api_version="2.2") -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/{api_version}/{path.lstrip('/')}"
    response = requests.post(url, headers=headers, data=json.dumps(payload), timeout=120)

    return {
        "ok": response.status_code in [200, 201],
        "status_code": response.status_code,
        "text": response.text[:5000],
        "json": response.json() if response.text and response.headers.get("content-type", "").startswith("application/json") else {}
    }


def workspace_path_exists(path: str) -> bool:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.0/workspace/get-status"
    response = requests.get(url, headers=headers, params={"path": path}, timeout=60)
    return response.status_code == 200


def normalize_depends_on(dep_task_keys):
    return [{"task_key": task_key} for task_key in dep_task_keys]


# ─────────────────────────────────────────────────────────────
# 1. Validate Notebook 32 exists
# ─────────────────────────────────────────────────────────────

notebook_32_exists = workspace_path_exists(ALERTING_NOTEBOOK_PATH)

print("Notebook 32 path:")
print(ALERTING_NOTEBOOK_PATH)
print("Notebook 32 exists:", notebook_32_exists)

if not notebook_32_exists:
    raise ValueError(
        f"Notebook 32 was not found at: {ALERTING_NOTEBOOK_PATH}. "
        "Confirm the notebook name is exactly 32_alerting_notification_routing."
    )

# ─────────────────────────────────────────────────────────────
# 2. Fetch current job settings
# ─────────────────────────────────────────────────────────────

job_get_result = api_get(
    path="jobs/get",
    params={"job_id": int(target_job_id)},
    api_version="2.1"
)

if not job_get_result["ok"]:
    raise ValueError(
        f"Could not fetch job settings. "
        f"HTTP {job_get_result['status_code']}: {job_get_result['text']}"
    )

job_payload = job_get_result["json"]
job_settings = job_payload.get("settings", {}) or {}
current_tasks = job_settings.get("tasks", []) or []

current_task_keys = [task.get("task_key") for task in current_tasks]

print("\nCurrent task count:", len(current_tasks))
print("Current task keys:")
print(json.dumps(current_task_keys, indent=2))

if "serving_health_check" not in current_task_keys:
    raise ValueError("Expected dependency task 'serving_health_check' was not found.")

if FINAL_SUMMARY_TASK_KEY not in current_task_keys:
    raise ValueError(f"Expected final task '{FINAL_SUMMARY_TASK_KEY}' was not found.")

# ─────────────────────────────────────────────────────────────
# 3. Build alerting task
# ─────────────────────────────────────────────────────────────

alerting_task = {
    "task_key": ALERTING_NOTEBOOK_TASK_KEY,
    "description": "Generate SupplySage alert events, route alerts through Delta-backed config, create alert inbox records, and log delivery attempts.",
    "depends_on": normalize_depends_on(ALERTING_DEPENDS_ON),
    "notebook_task": {
        "notebook_path": ALERTING_NOTEBOOK_PATH,
        "base_parameters": {
            "run_mode": "production",
            "alerting_mode": "scheduled_lakeflow",
            "send_enabled_default": "false"
        },
        "source": "WORKSPACE"
    },
    "timeout_seconds": 0
}

# ─────────────────────────────────────────────────────────────
# 4. Update final workflow summary dependency
# ─────────────────────────────────────────────────────────────
# Because Jobs API task arrays are merged by task_key, any existing task object
# we update should be sent as a full task object, not only the changed field.

workflow_summary_task = None

for task in current_tasks:
    if task.get("task_key") == FINAL_SUMMARY_TASK_KEY:
        workflow_summary_task = copy.deepcopy(task)
        break

existing_final_dependencies = [
    dep.get("task_key")
    for dep in workflow_summary_task.get("depends_on", [])
    if dep.get("task_key")
]

if ALERTING_NOTEBOOK_TASK_KEY not in existing_final_dependencies:
    updated_final_dependencies = existing_final_dependencies + [ALERTING_NOTEBOOK_TASK_KEY]
else:
    updated_final_dependencies = existing_final_dependencies

workflow_summary_task["depends_on"] = normalize_depends_on(updated_final_dependencies)

# ─────────────────────────────────────────────────────────────
# 5. Submit partial job update
# ─────────────────────────────────────────────────────────────

job_update_payload = {
    "job_id": int(target_job_id),
    "new_settings": {
        "tasks": [
            alerting_task,
            workflow_summary_task
        ],
        "tags": {
            **(job_settings.get("tags", {}) or {}),
            "alerting_enabled": "true",
            "alerting_version": "v1_config_driven_alert_routing"
        }
    }
}

print("\nJob update payload summary:")
print(json.dumps({
    "job_id": target_job_id,
    "tasks_to_upsert": [task["task_key"] for task in job_update_payload["new_settings"]["tasks"]],
    "new_alerting_task_depends_on": ALERTING_DEPENDS_ON,
    "workflow_summary_depends_on": updated_final_dependencies
}, indent=2))

if not UPDATE_JOB_WITH_ALERTING_TASK:
    raise ValueError("UPDATE_JOB_WITH_ALERTING_TASK is False. Set it to True to patch the job.")

update_result = api_post(
    path="jobs/update",
    payload=job_update_payload,
    api_version="2.2"
)

patch_finished_at = datetime.now(timezone.utc)

print("\nLakeflow job alerting patch result:")
print(json.dumps({
    "ok": update_result["ok"],
    "status_code": update_result["status_code"],
    "response_text": update_result["text"]
}, indent=2))

if not update_result["ok"]:
    raise ValueError(
        f"Failed to patch Lakeflow job with alerting task. "
        f"HTTP {update_result['status_code']}: {update_result['text']}"
    )

# ─────────────────────────────────────────────────────────────
# 6. Verify updated job
# ─────────────────────────────────────────────────────────────

verify_result = api_get(
    path="jobs/get",
    params={"job_id": int(target_job_id)},
    api_version="2.1"
)

if not verify_result["ok"]:
    raise ValueError(
        f"Could not verify updated job. "
        f"HTTP {verify_result['status_code']}: {verify_result['text']}"
    )

updated_job_payload = verify_result["json"]
updated_settings = updated_job_payload.get("settings", {}) or {}
updated_tasks = updated_settings.get("tasks", []) or []

updated_task_keys = [task.get("task_key") for task in updated_tasks]

alerting_task_after = [
    task for task in updated_tasks
    if task.get("task_key") == ALERTING_NOTEBOOK_TASK_KEY
]

workflow_summary_after = [
    task for task in updated_tasks
    if task.get("task_key") == FINAL_SUMMARY_TASK_KEY
]

alerting_task_found = len(alerting_task_after) == 1
workflow_summary_found = len(workflow_summary_after) == 1

workflow_summary_dependencies_after = []

if workflow_summary_found:
    workflow_summary_dependencies_after = [
        dep.get("task_key")
        for dep in workflow_summary_after[0].get("depends_on", [])
        if dep.get("task_key")
    ]

alerting_dependency_verified = False

if alerting_task_found:
    alerting_dependencies_after = [
        dep.get("task_key")
        for dep in alerting_task_after[0].get("depends_on", [])
        if dep.get("task_key")
    ]

    alerting_dependency_verified = all(
        dep in alerting_dependencies_after
        for dep in ALERTING_DEPENDS_ON
    )
else:
    alerting_dependencies_after = []

workflow_summary_depends_on_alerting = ALERTING_NOTEBOOK_TASK_KEY in workflow_summary_dependencies_after

patch_verification = {
    "workflow_deployment_id": workflow_deployment_id,
    "job_id": target_job_id,
    "job_name": updated_settings.get("name"),
    "job_url": f"{DATABRICKS_WORKSPACE_URL}/jobs/{target_job_id}",
    "notebook_32_path": ALERTING_NOTEBOOK_PATH,
    "notebook_32_exists": notebook_32_exists,
    "task_count_before": len(current_tasks),
    "task_count_after": len(updated_tasks),
    "alerting_task_key": ALERTING_NOTEBOOK_TASK_KEY,
    "alerting_task_found": alerting_task_found,
    "alerting_dependencies_after": alerting_dependencies_after,
    "alerting_dependency_verified": alerting_dependency_verified,
    "workflow_summary_depends_on_alerting": workflow_summary_depends_on_alerting,
    "workflow_summary_dependencies_after": workflow_summary_dependencies_after,
    "schedule_pause_status": (
        updated_settings.get("schedule", {}) or {}
    ).get("pause_status"),
    "patch_succeeded": bool(update_result["ok"]),
    "patch_verified": bool(
        update_result["ok"]
        and alerting_task_found
        and alerting_dependency_verified
        and workflow_summary_depends_on_alerting
    ),
    "patch_status": (
        "alerting_task_added_to_lakeflow_job"
        if update_result["ok"]
        and alerting_task_found
        and alerting_dependency_verified
        and workflow_summary_depends_on_alerting
        else "alerting_task_patch_verification_failed"
    ),
    "patch_started_at": patch_started_at.isoformat(),
    "patch_finished_at": patch_finished_at.isoformat()
}

print("\nAlerting task patch verification:")
print(json.dumps(patch_verification, indent=2))

# ─────────────────────────────────────────────────────────────
# 7. Save patch event
# ─────────────────────────────────────────────────────────────

ALERTING_PATCH_EVENTS_TABLE = f"{GOLD_SCHEMA}.gold_workflow_alerting_patch_events"

patch_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("job_id", StringType(), True),
    StructField("job_name", StringType(), True),
    StructField("job_url", StringType(), True),
    StructField("notebook_32_path", StringType(), True),
    StructField("notebook_32_exists", BooleanType(), True),
    StructField("task_count_before", IntegerType(), True),
    StructField("task_count_after", IntegerType(), True),
    StructField("alerting_task_key", StringType(), True),
    StructField("alerting_task_found", BooleanType(), True),
    StructField("alerting_dependency_verified", BooleanType(), True),
    StructField("workflow_summary_depends_on_alerting", BooleanType(), True),
    StructField("schedule_pause_status", StringType(), True),
    StructField("patch_succeeded", BooleanType(), True),
    StructField("patch_verified", BooleanType(), True),
    StructField("patch_status", StringType(), True),
    StructField("patch_payload_json", StringType(), True),
    StructField("patch_started_at", TimestampType(), True),
    StructField("patch_finished_at", TimestampType(), True),
])

patch_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "job_id": target_job_id,
    "job_name": updated_settings.get("name"),
    "job_url": f"{DATABRICKS_WORKSPACE_URL}/jobs/{target_job_id}",
    "notebook_32_path": ALERTING_NOTEBOOK_PATH,
    "notebook_32_exists": bool(notebook_32_exists),
    "task_count_before": int(len(current_tasks)),
    "task_count_after": int(len(updated_tasks)),
    "alerting_task_key": ALERTING_NOTEBOOK_TASK_KEY,
    "alerting_task_found": bool(alerting_task_found),
    "alerting_dependency_verified": bool(alerting_dependency_verified),
    "workflow_summary_depends_on_alerting": bool(workflow_summary_depends_on_alerting),
    "schedule_pause_status": (
        updated_settings.get("schedule", {}) or {}
    ).get("pause_status"),
    "patch_succeeded": bool(update_result["ok"]),
    "patch_verified": bool(patch_verification["patch_verified"]),
    "patch_status": patch_verification["patch_status"],
    "patch_payload_json": json.dumps(patch_verification, indent=2, default=str),
    "patch_started_at": patch_started_at,
    "patch_finished_at": patch_finished_at
}]

patch_df = spark.createDataFrame(patch_rows, schema=patch_schema)

(
    patch_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(ALERTING_PATCH_EVENTS_TABLE)
)

print(f"\nSaved alerting patch event to: {ALERTING_PATCH_EVENTS_TABLE}")

display(
    spark.table(ALERTING_PATCH_EVENTS_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_id",
        "task_count_before",
        "task_count_after",
        "alerting_task_found",
        "alerting_dependency_verified",
        "workflow_summary_depends_on_alerting",
        "schedule_pause_status",
        "patch_verified",
        "patch_status",
        "patch_finished_at"
    )
    .orderBy(F.desc("patch_finished_at"))
)

if patch_verification["patch_verified"]:
    print("\nNotebook 32 alerting is now wired into the scheduled Lakeflow Job.")
    print(f"Task count before: {len(current_tasks)}")
    print(f"Task count after: {len(updated_tasks)}")
    print(f"Schedule status: {patch_verification['schedule_pause_status']}")
else:
    raise ValueError("Alerting task patch did not verify cleanly.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Temporary Run Now Cell
# Trigger existing SupplySage Lakeflow job
# ─────────────────────────────────────────────────────────────

import requests
import json
from datetime import datetime, timezone

JOB_ID = 522140299855346

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

try:
    DATABRICKS_WORKSPACE_URL = ctx.apiUrl().get()
except Exception:
    DATABRICKS_WORKSPACE_URL = None

try:
    DATABRICKS_TOKEN = ctx.apiToken().get()
except Exception:
    DATABRICKS_TOKEN = None

assert DATABRICKS_WORKSPACE_URL, "DATABRICKS_WORKSPACE_URL missing."
assert DATABRICKS_TOKEN, "DATABRICKS_TOKEN missing."

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

run_now_url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/run-now"

payload = {
    "job_id": JOB_ID
}

response = requests.post(
    run_now_url,
    headers=headers,
    data=json.dumps(payload),
    timeout=60
)

print("Status code:", response.status_code)
print(response.text)

response.raise_for_status()

run_now_result = response.json()
NEW_RUN_ID = run_now_result["run_id"]

print("\n✅ Triggered new Lakeflow run")
print("Job ID:", JOB_ID)
print("New run ID:", NEW_RUN_ID)
print("Triggered at UTC:", datetime.now(timezone.utc).isoformat())

In [0]:
# ─────────────────────────────────────────────────────────────
# Inspect failed Lakeflow run task errors
# ─────────────────────────────────────────────────────────────

import requests
import json
from datetime import datetime, timezone

RUN_ID = 365541620793872

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_WORKSPACE_URL = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

run_get_url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/runs/get"

response = requests.get(
    run_get_url,
    headers=headers,
    params={"run_id": RUN_ID},
    timeout=60
)

print("Status code:", response.status_code)
response.raise_for_status()

run_info = response.json()

print("\nRun state:")
print(json.dumps(run_info.get("state", {}), indent=2))

tasks = run_info.get("tasks", [])

failed_or_problem_tasks = []

print("\nTask states:")
for task in tasks:
    task_key = task.get("task_key")
    task_run_id = task.get("run_id")
    task_state = task.get("state", {})
    life_cycle = task_state.get("life_cycle_state")
    result = task_state.get("result_state")
    message = task_state.get("state_message")

    print(f"{task_key} | task_run_id={task_run_id} | life_cycle={life_cycle} | result={result} | message={message}")

    if result in ["FAILED", "TIMEDOUT", "CANCELED", "UPSTREAM_FAILED"]:
        failed_or_problem_tasks.append(task)

print("\nProblem tasks:")
for task in failed_or_problem_tasks:
    print(
        task.get("task_key"),
        "| task_run_id:",
        task.get("run_id"),
        "| state:",
        task.get("state", {})
    )

print("\nChecked at UTC:", datetime.now(timezone.utc).isoformat())

In [0]:
# ─────────────────────────────────────────────────────────────
# Get output/error for failed Notebook 14 task runs
# ─────────────────────────────────────────────────────────────

import requests
import json
from datetime import datetime, timezone

FAILED_TASK_RUN_IDS = [
    484892836481788,
    618673987086423
]

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_WORKSPACE_URL = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

get_output_url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/runs/get-output"

for task_run_id in FAILED_TASK_RUN_IDS:
    print("\n" + "=" * 100)
    print("Inspecting task run ID:", task_run_id)
    print("=" * 100)

    response = requests.get(
        get_output_url,
        headers=headers,
        params={"run_id": task_run_id},
        timeout=60
    )

    print("Status code:", response.status_code)

    try:
        response.raise_for_status()
        output = response.json()

        print("\nNotebook output:")
        print(output.get("notebook_output", {}).get("result", ""))

        print("\nError:")
        print(output.get("error", ""))

        print("\nMetadata keys:")
        print(output.keys())

    except Exception as e:
        print("Failed to fetch output:", repr(e))
        print(response.text[:4000])

print("\nChecked at UTC:", datetime.now(timezone.utc).isoformat())